# Exploratory data analysis - 30-day mortality after spinal trauma

This notebook documents how the analysis dataset was assembled and checked, and how the final
set of candidate predictors was chosen, for a model predicting death within 30 days of a spinal
trauma consultation.

It runs top to bottom, from the raw registry export to the final feature list. Each section
states what it examines, shows the result, and records the decision that follows from it.

**Data.** A single SPSS export from the trauma registry: 1,517 records, of which 1,486 meet the
study inclusion criteria. Variable names come from the registry and are mostly Dutch or
abbreviated; each is explained where it is first used, and section 2 compares the ones that
measure the same thing.

**Outcome.** `D30_Mortality` - died within 30 days of the accident (1) or not (0).

## Contents

- [0. Variables in this dataset](#sec0)
- [1. Loading the data and defining the cohort](#sec1)
    - [1b. Sentinel values (888 and 999)](#sec1b)
- [2. Variable comparison - choosing between overlapping features](#sec2)
    - [2a. Comorbidity: `COMORB` vs `ASA` / `ASAcat` vs binary codes](#sec2a)
    - [2b. Frailty: `MFI5` vs `FrailtyCat`](#sec2b)
    - [2c. TBI: `TBI_CT` vs `TBI_totaal`](#sec2c)
    - [2d. Spinal cord injury: `SCI_yes_no` vs `ASIA` vs `N_CODE`](#sec2d)
    - [2e. Injury mechanism: `OORZAAKCATEGORIEID` vs `INTENTIE`](#sec2e)
    - [2f. AO Spine injury type: `TYPE_A` / `TYPE_B` / `TYPE_C` / `TYPE_F_INJURY`](#sec2f)
    - [2g. Sex: `Geslacht01` vs `MAN_VROUW`](#sec2g)
- [3. Missingness](#sec3)
    - [3b. Missingness mechanism](#sec3b)
- [4. Data integrity checks](#sec4)
- [5. Target distribution](#sec5)
- [6. Predictor distributions](#sec6)
    - [6b. Normality tests — scaler decision](#sec6b)
    - [6c. Injury mechanism (`INTENTIE`) - raw codes before grouping](#sec6c)
    - [6d. `MFI5` - collapsing the sparse top grades](#sec6d)
    - [6e. `NUMBER_OF_FRACTURES` - a count, capped at 6+](#sec6e)
- [7. Spinal region involvement](#sec7)
    - [7b. Number of CTLS regions involved (`n_regions`)](#sec7b)
- [8. Neurological status: `N_CODE`, `ASIA`, and the not-examinable state](#sec8)
- [9. Collinearity](#sec9)
    - [9b. Collinearity within the final predictor set](#sec9b)
- [10. Enrollment over time](#sec10)
- [11. Pilot multivariate logistic regression](#sec11)
    - [11b. Cross-validated R²_cs — go/no-go check](#sec11b)
    - [11c. Functional form - is a linear term adequate for age and BMI?](#sec11c)
- [12. Summary and conclusions](#sec12)

---

In [ ]:
import pyreadstat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Suppresses pandas/sklearn deprecation and future warnings, which are numerous here and would
# otherwise bury the printed output. Errors are not affected.
warnings.filterwarnings("ignore")

<a id="sec0"></a>

## 0. Variables in this dataset

Column names come from the trauma registry and are mostly Dutch or abbreviated. This table is
the reference for the rest of the notebook; sections below use the registry names directly.

Codings marked *(codebook)* are taken from the SPSS value labels exported in section 1.

### Outcome and timing

| Variable | Meaning | Coding |
|---|---|---|
| `D30_Mortality` | **Outcome.** Died within 30 days of the accident | 1 = died, 0 = alive *(codebook)* |
| `Y1_MORTALITY` | Died within 1 year of the accident. | 1 = died, 0 = alive *(codebook)* |
| `DATUM_ONGEVAL` | Date of accident - the index date from which the 30 days are counted | SPSS date (seconds since 1582-10-14) |
| `DATUM_OVERLEDEN` | Date of death, where applicable | SPSS date |
| `INCLUSION` | Registry flag for records meeting the study criteria | 1 = include |

### Demographics

| Variable | Meaning | Coding |
|---|---|---|
| `LEEFTIJD_EXACT` | Age at accident (*leeftijd* = age) | years, continuous |
| `Geslacht01` | Sex (*geslacht* = sex) | 0 / 1 |
| `BMI` | Body mass index | kg/m2 |

### Comorbidity and frailty

| Variable | Meaning | Coding |
|---|---|---|
| `COMORB` | **ASA physical status grade** - a global fitness-for-anaesthesia judgement. The name is misleading | 1 = normally healthy, 2 = mild systemic disease, 3 = severe systemic disease, 4-5 = life-threatening / moribund *(codebook)* |
| `ASA` | Free-text version of the same grade, entered inconsistently | text; largely blank or `GEEN`/`NEE` |
| `ASAcat` | `COMORB` collapsed into two groups | 0 = ASA 1-2, 1 = ASA 3-5 *(codebook)* |
| `MFI5` | 5-item modified Frailty Index | 0-5, count of items present |
| `FrailtyCat` | `MFI5` dichotomised | 0 = not frail, 1 = pre-frail/frail *(codebook)* |
| `HT_CODE` | Hypertension - an `MFI5` item | 0 / 1 |
| `CHF_CODE` | Congestive heart failure - an `MFI5` item | 0 / 1 |
| `COPD_CODE` | Chronic obstructive pulmonary disease - an `MFI5` item | 0 / 1 |
| `DM_CODE` | Diabetes mellitus - an `MFI5` item | 0 / 1 |
| `DEPENDENT_CODE` | Functionally dependent - an `MFI5` item | 0 / 1 |

### Injury severity and head injury

| Variable | Meaning | Coding |
|---|---|---|
| `ISS08` | Injury Severity Score - overall anatomical severity across body regions | 1-75, higher = more severe |
| `TBI_CT` | Traumatic brain injury confirmed on CT | 0 = no intracranial abnormality, 1 = intracranial abnormality *(codebook)* |
| `TBI_totaal` | Any traumatic brain injury, by broader criteria (*totaal* = total) | 0 / 1 |

### Neurological status

| Variable | Meaning | Coding |
|---|---|---|
| `ASIA` | ASIA Impairment Scale - severity of spinal cord injury. **Higher is better** | 1 = A complete, 2 = B sensory incomplete, 3 = C and 4 = D motor incomplete, 5 = E normal; 0 = not determinable *(codebook)* |
| `N_CODE` | AO Spine neurological modifier. **Higher is worse** | 1 = N0 intact, 2 = N1 transient, 3 = N2 radicular, 4 = N3 incomplete cord/cauda equina, 5 = N4 complete cord, 6 = NX not examinable *(codebook)* |
| `SCI_yes_no` | Spinal cord injury present | 0 = no, 1 = yes *(codebook)* |

### Fracture pattern

| Variable | Meaning | Coding |
|---|---|---|
| `NUMBER_OF_FRACTURES` | Number of spinal fractures | count (0 where no fracture was classified) |
| `AMOUNT_OF_LEVELS_INJURED` | Number of vertebral levels involved | count |
| `TYPE_A_INJURY` | AO Spine type A - compression injury | 0 / 1 *(codebook: Nee/Ja)* |
| `TYPE_B_INJURY` | AO Spine type B - tension-band injury | 0 / 1 |
| `TYPE_C_INJURY` | AO Spine type C - translation/displacement injury | 0 / 1 |
| `TYPE_F_INJURY` | AO Spine type F - facet joint injury | 0 / 1 |
| `C`, `T`, `L`, `S` | Cervical / thoracic / lumbar / sacral region involved. Not mutually exclusive | `JA` (yes) / `NEE` (no); converted to 1/0 in section 7 |

### Injury mechanism

| Variable | Meaning | Coding |
|---|---|---|
| `INTENTIE` | Detailed mechanism of injury | 14 codes, listed with labels in section 6c |
| `OORZAAKCATEGORIEID` | Broad cause category | 2 = violence, 4 = traffic, 5 = occupational, 6 = private/domestic, 8 = sport, 9 = self-harm, 11 = other *(codebook)* |

### Variables derived in this notebook

| Variable | Defined in | Meaning |
|---|---|---|
| `INTENTIE_cat` | 6c | `INTENTIE` grouped into 6 clinical categories: `MVA` (motorised vehicle accident), `OVA` (other vehicle accident - cyclist, pedestrian), `violence`, `fall_low` (same-level fall), `fall_high` (fall from height), `other` |
| `n_regions` | 7b | Number of spinal regions involved, 0-4, from `C`/`T`/`L`/`S` |
| `NEURO_NOT_EXAMINABLE` | 8 | 1 where `ASIA` was not recorded, i.e. the neurological exam could not be performed |
| `ASIA_exam` | 8 | `ASIA` restricted to patients who could be examined |
| `N_CODE_valid` | 8 | `N_CODE` restricted to its documented N0-N4 range |
| `survival_days` | 5 | Days from accident to death |
| `year` | 8 | Calendar year of the accident |
---

<a id="sec1"></a>

## 1. Loading the data and defining the cohort

The source is a single SPSS file exported from the trauma registry, holding one row per patient
consulted for spinal trauma. The registry marks which records belong in this study with an
`INCLUSION` flag; everything below uses only those rows.

Two objects are kept throughout:

- `df_raw` - every row in the export, before any filtering
- `df` - the analysis cohort (`INCLUSION == 1`), which all later sections use

**Index date.** The outcome is death within 30 days **of the accident**, not of admission or of
consultation: `survival_days` in section 5 is computed from `DATUM_ONGEVAL`.

The study setting, the inclusion and exclusion criteria behind the `INCLUSION` flag, and the
reason 31 records were excluded are described in the accompanying manuscript; this notebook takes
the flag as given.

In [ ]:
# The same file the modelling pipeline reads (config/pipeline.yaml -> data.raw_dir).
# Path is relative to the repository so the notebook and the pipeline cannot drift apart.
from pathlib import Path

# The registry export itself is not part of this repository.
DATA_PATH = next(Path("../data/raw").glob("*.sav"))

df_raw, meta = pyreadstat.read_sav(str(DATA_PATH))

# `meta` carries the SPSS metadata, including the code -> text mapping for every
# categorical variable. It is used in the next cells and again in section 6c.
df = df_raw[df_raw["INCLUSION"] == 1.0].copy()

print(f"Rows in export    : {len(df_raw)}")
print(f"Included patients : {len(df)}")
print(f"Excluded          : {len(df_raw) - len(df)}")
print(f"Columns           : {df.shape[1]}")

In [ ]:
# The SPSS file ships a codebook: for each categorical variable, what each numeric code means.
# Saving it to data/schemas/value_labels.json gives a lookup for the coded variables used below
# (INTENTIE in section 6c, for example) and makes the codes readable outside SPSS.
import json, pathlib

labels_serialisable = {
    col: {str(k): v for k, v in mapping.items()}
    for col, mapping in meta.variable_value_labels.items()
}
out_path = pathlib.Path("../data/schemas/value_labels.json")
out_path.write_text(json.dumps(labels_serialisable, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved code -> label mappings for {len(labels_serialisable)} variables to {out_path}")

<a id="sec1b"></a>

### 1b. Sentinel values (888 and 999)

A **sentinel value** is a real number stored inside a data field to mean *no value here*
rather than an actual measurement. This registry uses two, both declared in the SPSS codebook shipped with the file:

| Code | SPSS label (Dutch) | Meaning |
|---|---|---|
| 888 | *Naar waarde gezocht maar niet gevonden* | the value was looked for but could not be found |
| 999 | *Niet ingevuld en nog na te zoeken* | never filled in, still to be looked up |

Both mean the value is unknown. Before changing anything, the cell below shows exactly where
they occur and whether the codebook declares them for that column.

In [ ]:
# Where do the sentinel codes occur, and are they declared in the SPSS codebook?
rows = []
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    n888, n999 = int((df[col] == 888).sum()), int((df[col] == 999).sum())
    if n888 or n999:
        declared = meta.variable_value_labels.get(col, {})
        rows.append({
            "column": col,
            "n_888": n888,
            "n_999": n999,
            "pct_of_cohort": round((n888 + n999) / len(df) * 100, 1),
            "declared_in_codebook": "yes" if (888.0 in declared or 999.0 in declared) else "no",
        })

sentinels = pd.DataFrame(rows).sort_values("pct_of_cohort", ascending=False)
print("Columns containing sentinel codes, before replacement:")
print(sentinels.to_string(index=False))

Two observations before replacing them.

`ASIA` (a spinal cord injury severity grade, explained in section 2d) accounts for most of the
sentinel values - 117 patients, 7.9% of the cohort - and is the one column where 888 is
**not** declared in the codebook: the convention was applied without being documented.
Section 8 shows that these are not ordinary missing values at all, and that treating them as
such would discard the single strongest predictor in the dataset.

`ID` carries one 888 and one 999, so two patients are left without an identifier. That is why
the integrity check in section 4 reports two missing IDs.

Both codes are now replaced with `NaN` (the standard marker for a missing value in pandas)
across the whole table, so that every later step - missingness counts, plots, imputation -
treats them as missing rather than as measurements.

In [ ]:
df.replace({888: np.nan, 999: np.nan}, inplace=True)

num = df.select_dtypes(include=[np.number])
remaining = int(((num == 888) | (num == 999)).sum().sum())
print(f"Sentinel codes remaining after replacement: {remaining}")
print(f"Cells now marked missing: {int(df.isna().sum().sum())}")

<a id="sec2"></a>

## 2. Variable comparison - choosing between overlapping features

The dataset carries several pairs or clusters of columns measuring the same construct. This
section compares them so the pipeline keeps one of each. The decisions are summarised at the end
and are what `features:` in `config/pipeline.yaml` implements.

How much data is missing for each of these variables is quantified in section 3.

<a id="sec2a"></a>

### 2a. Comorbidity: `COMORB` vs `ASA` / `ASAcat` vs binary codes

Four overlapping ways of recording pre-existing illness:

- **`COMORB`** - despite the name, this is the **ASA physical status grade** (American Society
  of Anesthesiologists), 1-5, per the SPSS codebook: 1 = normally healthy, 2 = mild systemic
  disease, 3 = severe systemic disease, and so on.
- **`ASA`** - a free-text column holding the same idea, but entered inconsistently.
- **`ASAcat`** - `COMORB` collapsed to two groups, labelled *ASA 1-2* vs *ASA 3-5*.
- **Five binary flags** - `HT_CODE` (hypertension), `CHF_CODE` (congestive heart failure),
  `COPD_CODE` (chronic obstructive pulmonary disease), `DM_CODE` (diabetes mellitus) and
  `DEPENDENT_CODE` (functionally dependent), which are the five items of the modified Frailty
  Index (`MFI5`, section 2b).

In [ ]:
comorb_cols = ["COMORB", "ASA", "ASAcat", "HT_CODE", "CHF_CODE", "COPD_CODE", "DM_CODE", "DEPENDENT_CODE"]

print("=== Missingness ===")
print(df[comorb_cols].isnull().mean().round(3).to_string())

print("\n=== COMORB value counts ===")
print(df["COMORB"].value_counts(dropna=False))

print("\n=== ASA value counts ===")
print(df["ASA"].value_counts(dropna=False))

print("\n=== ASAcat value counts ===")
print(df["ASAcat"].value_counts(dropna=False))

In [ ]:
# Cross-tab: COMORB vs ASA — how well do they agree?
# Recode ASA to numeric for comparison
df["ASA_num"] = pd.to_numeric(df["ASA"], errors="coerce")
print(pd.crosstab(df["COMORB"], df["ASA_num"], margins=True, dropna=False))

In [ ]:
# How many patients have ≥1 binary comorbidity code = 1?
bin_codes = ["HT_CODE", "CHF_CODE", "COPD_CODE", "DM_CODE", "DEPENDENT_CODE"]
df["any_comorbidity_code"] = df[bin_codes].eq(1.0).any(axis=1)
print(df["any_comorbidity_code"].value_counts())

# Do patients with COMORB > 1 align with having a binary code?
print("\nCOMORB>1 vs any binary code:")
print(pd.crosstab(df["COMORB"] > 1, df["any_comorbidity_code"], dropna=False))

Two things in the output above matter for the choice.

**`ASA` is unusable.** It is a text column: 747 of 1,486 entries are blank and a further 390 read
`GEEN` (none) or `NEE` (no), leaving roughly 344 actual grades. Because the blanks are empty
strings rather than true nulls, the missingness table in section 3 reports `ASA` as 0% missing,
which is misleading - the information is absent, just not recorded as absent.

**`COMORB` and the binary flags disagree substantially.** 604 patients have `COMORB` above 1
(some systemic disease) yet none of the five flags set, and 11 have a flag set with `COMORB` = 1.
They are not two measurements of one quantity: the flags capture five specific diagnoses, while
`COMORB` is a global fitness-for-anaesthesia judgement.

<a id="sec2b"></a>

### 2b. Frailty: `MFI5` vs `FrailtyCat`

In [ ]:
print("=== Missingness ===")
print(df[["MFI5", "FrailtyCat"]].isnull().mean().round(3))

print("\n=== MFI5 distribution ===")
print(df["MFI5"].value_counts(dropna=False).sort_index())

print("\n=== FrailtyCat distribution ===")
print(df["FrailtyCat"].value_counts(dropna=False))

# Verify FrailtyCat is derived from MFI5
print("\nMFI5 vs FrailtyCat cross-tab:")
print(pd.crosstab(df["MFI5"], df["FrailtyCat"], dropna=False))

`FrailtyCat` is `MFI5 >= 1` dichotomised, near-exactly (3/1,486 discordant), so it carries less
information than `MFI5` itself and is dropped.

The five binary codes (`HT_CODE`, `CHF_CODE`, `COPD_CODE`, `DM_CODE`, `DEPENDENT_CODE`) are the
*items* of `MFI5` - it is their sum. Keeping both the index and its components would enter the
same information twice, so the index is kept and the five items dropped. Note that the items are
**not** strongly correlated with one another (maximum |r| = 0.30, section 9): they are dropped
because they are redundant *with the index*, not with each other.

`MFI5` is retained as a candidate predictor alongside `COMORB`. They are only weakly correlated
(Spearman 0.38) and measure different things - accumulated frailty deficits versus a global
fitness-for-anaesthesia grade - so neither substitutes for the other. `MFI5` also has no missing
values, where `COMORB` has 3.6%. As `MFI5` 4 and 5 had low frequency, we collapsed them into 3/4/5 together.

<a id="sec2c"></a>

### 2c. TBI: `TBI_CT` vs `TBI_totaal`

In [ ]:
print("=== Missingness ===")
print(df[["TBI_CT", "TBI_totaal"]].isnull().mean().round(3))

print("\n=== Cross-tab ===")
print(pd.crosstab(df["TBI_CT"], df["TBI_totaal"], margins=True, dropna=False))
# Expectation: TBI_totaal >= TBI_CT (totaal is broader definition)

The expectation holds: `TBI_totaal` is a superset of `TBI_CT`. No patient is CT-positive but
`TBI_totaal`-negative, and 127 patients are `TBI_totaal`-positive without CT-confirmed
intracranial abnormality - clinically suspected or otherwise diagnosed head injury.

In discussion with the clinical team `TBI_CT` is kept because it is the objective, reproducibly defined variable: an abnormality
visible on CT. `TBI_totaal` is broader but its 127 extra cases rest on a mix of criteria that
cannot be reconstructed from the export.

<a id="sec2d"></a>

### 2d. Spinal cord injury: `SCI_yes_no` vs `ASIA` vs `N_CODE`

First look at the three overlapping neurological columns. The `N_CODE` defects and the
not-examinable state are diagnosed in detail in **section 8**, which is where the final
keep/drop decision is made.

In [ ]:
sci_cols = ["SCI_yes_no", "ASIA", "N_CODE"]
print("=== Missingness ===")
print(df[sci_cols].isnull().mean().round(3))

print("\n=== ASIA value counts ===")
print(df["ASIA"].value_counts(dropna=False).sort_index())

print("\n=== N_CODE value counts ===")
print(df["N_CODE"].value_counts(dropna=False).sort_index())

print("\n=== SCI_yes_no vs ASIA ===")
print(pd.crosstab(df["SCI_yes_no"], df["ASIA"], dropna=False))

**How to read these scales** (from the SPSS codebook):

- `ASIA` - ASIA Impairment Scale, the standard grading of spinal cord injury severity:
  **1 = A** (complete), **2 = B** (sensory incomplete), **3 = C** and **4 = D** (motor
  incomplete), **5 = E** (normal). **0 = not determinable.** Note the direction: *higher is
  better*.
- `N_CODE` - the AO Spine neurological modifier: **1 = N0** (intact), **2 = N1** (transient
  deficit), **3 = N2** (radicular symptoms), **4 = N3** (incomplete cord or cauda equina),
  **5 = N4** (complete cord injury), **6 = NX** (patient could not be examined). Direction is
  the reverse of `ASIA`: *higher is worse*.
- `SCI_yes_no` - a simple yes/no flag for spinal cord injury.

The final decision on this group is deferred to **section 8**, which shows that `N_CODE` carries
a defect not visible here.

<a id="sec2e"></a>

### 2e. Injury mechanism: `OORZAAKCATEGORIEID` vs `INTENTIE`

In [ ]:
print("=== Missingness ===")
print(df[["OORZAAKCATEGORIEID", "INTENTIE"]].isnull().mean().round(3))

print()
print("=== OORZAAKCATEGORIEID counts (with SPSS labels) ===")
oorzaak_labels = meta.variable_value_labels.get("OORZAAKCATEGORIEID", {})
for code, n in df["OORZAAKCATEGORIEID"].value_counts(dropna=False).sort_index().items():
    label = oorzaak_labels.get(code, "?") if pd.notna(code) else "missing"
    print(f"  {str(code):>6}  {label:<36}  n={n}")

print()
print("=== INTENTIE counts ===")
print(df["INTENTIE"].value_counts(dropna=False))

# Do the two nest, or do they cross? Rows 10 (same-level fall) and 11 (fall from
# height) are the ones that decide it - see the conclusion below.
print()
print("=== INTENTIE x OORZAAKCATEGORIEID ===")
print(pd.crosstab(df["INTENTIE"], df["OORZAAKCATEGORIEID"], dropna=False))

print()
print("Median ISS08 by fall type: "
      f"code 10 (same level) = {df.loc[df['INTENTIE'] == 10, 'ISS08'].median():.0f}, "
      f"code 11 (from height) = {df.loc[df['INTENTIE'] == 11, 'ISS08'].median():.0f}")


The two columns record different things, not one thing at two resolutions.

`OORZAAKCATEGORIEID` is the **setting** the injury occurred in - its SPSS labels are *Verkeer*
(traffic, n=565), *Prive* (domestic, n=538), *Bedrijfsongeval* (occupational, n=166),
*Zelfmutilatie/TS* (self-harm, n=109), *Sport* (n=69) and *geweld* (violence, n=29). `INTENTIE`
records the **mechanism** - what physically happened to the patient.

The crosstab shows the two cross rather than nest. The 487 same-level falls (`INTENTIE` 10) are
spread over *Prive* (403), *Bedrijfsongeval* (43), *Sport* (31) and *Zelfmutilatie* (8), and the
307 falls from height (`INTENTIE` 11) over *Prive* (111), *Zelfmutilatie* (88), *Bedrijfsongeval*
(78) and *Sport* (21). The largest `OORZAAK` category, *Prive*, therefore merges 403 same-level
falls with 111 falls from height.

That merge is the reason for the drop: the two fall types differ in injury severity (median
`ISS08` 16 vs 22, printed above), `INTENTIE_cat` separates them (section 6c) and
`OORZAAKCATEGORIEID` does not. **Kept: `INTENTIE`. Dropped: `OORZAAKCATEGORIEID`.**

One caveat, recorded rather than acted on: `OORZAAK` is not purely redundant. Its
*Zelfmutilatie/TS* category identifies 109 patients whose injury was self-inflicted - intent,
which `INTENTIE` does not encode anywhere. It is dropped because it cannot substitute for
mechanism, not because it contains nothing of its own.


<a id="sec2f"></a>

### 2f. AO Spine injury type: `TYPE_A` / `TYPE_B` / `TYPE_C` / `TYPE_F_INJURY`

In [ ]:
injury_type_cols = ["TYPE_A_INJURY", "TYPE_B_INJURY", "TYPE_C_INJURY", "TYPE_F_INJURY"]

print("=== Prevalence per type ===")
for col in injury_type_cols:
    n = df[col].eq(1).sum()
    print(f"  {col}: {n} ({n/len(df):.1%})")

# Number of injury types per patient
df["n_injury_types"] = df[injury_type_cols].eq(1).sum(axis=1)
print("\n=== Number of injury types per patient ===")
print(df["n_injury_types"].value_counts().sort_index().to_string())
n_multi = (df["n_injury_types"] > 1).sum()
print(f"\nPatients with >1 type: {n_multi} ({n_multi/len(df):.1%})")

print("\n=== Co-occurrence matrix (counts with both types = 1) ===")
co = df[injury_type_cols].eq(1).astype(int)
print(co.T.dot(co).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: stacked bar — prevalence
counts_pos = df[injury_type_cols].eq(1).sum()
counts_neg = df[injury_type_cols].eq(0).sum()
counts_miss = df[injury_type_cols].isnull().sum()
bottom_neg = counts_pos
bottom_miss = counts_pos + counts_neg
axes[0].bar(injury_type_cols, counts_pos, label="1 (present)", color="steelblue")
axes[0].bar(injury_type_cols, counts_neg, bottom=counts_pos, label="0 (absent)", color="lightsteelblue")
axes[0].bar(injury_type_cols, counts_miss, bottom=bottom_miss, label="missing", color="lightgray")
axes[0].set_ylabel("Patients")
axes[0].set_title("Injury type prevalence")
axes[0].legend()

# Right: distribution of number of concurrent types
vc = df["n_injury_types"].value_counts().sort_index()
axes[1].bar(vc.index.astype(str), vc.values, color="steelblue")
axes[1].set_xlabel("Number of concurrent injury types")
axes[1].set_ylabel("Patients")
axes[1].set_title("How many types per patient?")
for i, (x, v) in enumerate(zip(vc.index, vc.values)):
    axes[1].text(i, v + 5, str(v), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("../outputs/injury_type_distribution.png", dpi=150)
plt.show()

The AO Spine classification describes fracture morphology: **type A** compression injuries,
**type B** tension-band (posterior or anterior) injuries, **type C** displacement/translation
injuries, and **type F** facet joint injuries. They are recorded as four independent yes/no
flags rather than one category because a patient can sustain more than one type, and 385
patients (25.9%) do.

**56 patients (3.8%) carry none of the four flags**, and the same 56 have
`NUMBER_OF_FRACTURES` = 0. These are spinal trauma consultations without a classifiable
fracture. They are retained: a
consultation without a fracture is a legitimate member of the cohort, and all four flags being
zero encodes that correctly.

<a id="sec2g"></a>
### 2g. Sex: `Geslacht01` vs `MAN_VROUW`

The export carries sex twice: `MAN_VROUW` as the registry's original string (`M` / `V`, Dutch
*man* / *vrouw*) and `Geslacht01` as a numeric recode. Nothing in the export states which
numeric code means which sex, so the direction has to be established from the data itself.

In [ ]:
print("=== Missingness (included cohort) ===")
print(df[["Geslacht01", "MAN_VROUW"]].isnull().mean().round(3))

print("\n=== Value counts, full export (before the inclusion filter) ===")
print(df_raw["MAN_VROUW"].value_counts(dropna=False))
print(df_raw["Geslacht01"].value_counts(dropna=False))

print("\n=== Cross-tab, full export ===")
print(pd.crosstab(df_raw["MAN_VROUW"].fillna("<missing>"),
                  df_raw["Geslacht01"].fillna(-1),
                  margins=True, dropna=False))

# Does the string recode onto the numeric column without exception?
recode = df_raw["MAN_VROUW"].str.upper().map({"M": 0.0, "V": 1.0})
mismatch = df_raw[recode != df_raw["Geslacht01"]]
print(f"\n=== Rows where the M->0 / V->1 recode does not hold: {len(mismatch)} ===")
print(mismatch[["MAN_VROUW", "Geslacht01", "INCLUSION"]])

print("\n=== 30-day mortality by sex (included cohort) ===")
print(df.groupby("MAN_VROUW")["D30_Mortality"].agg(["size", "sum", "mean"]).round(3))

The two columns hold the same information. Across all 1,517 exported records the mapping is
exactly **`M` -> 0 and `V` -> 1**, with one exception: a single record entered as lowercase
`m` was missed by the (case-sensitive) recode and left `Geslacht01` missing. That record has
`INCLUSION = 0`, so the analysis cohort of 1,486 has complete sex data and the two columns
agree on every included patient.

**`Geslacht01` = 0 is male and 1 is female**, not the reverse. This matters because the
cohort is male-predominant (1,066 male vs 420 female), which is what a spinal trauma
population should look like - reading the codes the other way round would have described a
71% female trauma cohort. 30-day mortality is near-identical between the two groups
(13.4% male vs 13.1% female), so the direction of the coding cannot be recovered from the
outcome; it rests on `MAN_VROUW`.

`Geslacht01` is kept as the model predictor: it is already numeric and needs no encoding.
`MAN_VROUW` is dropped from the feature set but is the reason the coding direction is
documented rather than assumed. Every downstream label - the schema comment, the SHAP
display name in `config/pipeline.yaml`, and the sex panel of the subgroup instability plot -
follows the direction established here.

### Variable selection decisions - section 2 summary

| Group | Keep | Drop | Reason |
|---|---|---|---|
| Comorbidity | `COMORB` (ASA grade 1-5) | `ASA`, `ASAcat` | `ASA` is a text column that is blank or non-informative for 1,137/1,486 patients. `ASAcat` is `COMORB` collapsed to ASA 1-2 vs 3-5, so it carries strictly less information |
| Frailty | `MFI5` (0-5) | `FrailtyCat`, `HT_CODE`, `CHF_CODE`, `COPD_CODE`, `DM_CODE`, `DEPENDENT_CODE` | The five codes are the items `MFI5` is built from, so the index replaces them. `FrailtyCat` is `MFI5 >= 1` dichotomised, near-exactly (3/1,486 discordant), and drops ordinality. `MFI5` is kept alongside `COMORB`: they correlate only weakly (rho 0.38) and capture different constructs - see 2b |
| TBI | `TBI_CT` | `TBI_totaal` | CT-confirmed is the specific, objectively verifiable definition; `TBI_totaal` is broader but less specific |
| Spinal cord injury | `ASIA` + derived `NEURO_NOT_EXAMINABLE` | `SCI_yes_no`, `N_CODE` | All three measure one construct (r = -0.96 / -0.93 / 0.90, section 9). `ASIA` is the most granular; `N_CODE` additionally carries a 2019-2020 collection gap. See section 8 |
| Injury mechanism | `INTENTIE_cat` (6 groups, section 6c) | `OORZAAKCATEGORIEID` | Different constructs, not two resolutions of one: `OORZAAK` is the *setting* (traffic / domestic / occupational / sport / self-harm), `INTENTIE` the *mechanism*. `OORZAAK`'s largest category merges 403 same-level falls with 111 falls from height, a distinction that tracks severity - see 2e |
| AO injury type | all four `TYPE_*_INJURY` flags | - | Not mutually exclusive and not redundant; kept as separate binaries |
| Sex | `Geslacht01` (**0 = male, 1 = female**) | `MAN_VROUW` | The same variable twice: `MAN_VROUW` (`M`/`V`) recodes 1:1 onto `Geslacht01`. The numeric column is model-ready; the string column is what establishes which code is which - see 2g |

The five binary comorbidity flags are **not** strongly correlated with one another (maximum
|r| = 0.30). They are dropped because each is an item of `MFI5`, which is retained in their
place - not because they duplicate one another.

> This table is what `config/pipeline.yaml` implements. Where a decision was revised during the
> analysis, the table records the final choice.

`DATUM_OVERLEDEN` is excluded from all modelling; only `D30_Mortality` is used as the outcome.

<a id="sec3"></a>

## 3. Missingness

Percentage missing for every schema column, after sentinels 888/999 have been set to NaN.
The 30% line is the threshold above which a variable would be excluded on missingness grounds
alone; nothing approaches it.

Note that `ASIA`'s missingness is not ordinary missingness - it marks patients whose neurological
exam could not be performed. Section 8 diagnoses this and converts it into an explicit indicator.

In [ ]:
SCHEMA_COLS = [
    # identifiers / targets
    "ID", "Y1_MORTALITY", "DATUM_ONGEVAL", "D30_Mortality",
    # demographics
    "LEEFTIJD_EXACT", "Geslacht01", "BMI",
    # comorbidity
    "ASA", "MFI5", "HT_CODE", "CHF_CODE", "COPD_CODE", "DM_CODE", "DEPENDENT_CODE",
    "COMORB", "ASAcat", "FrailtyCat",
    # injury severity
    "ISS08",
    # spinal cord
    "ASIA", "SCI_yes_no", "N_CODE",
    # fractures
    "NUMBER_OF_FRACTURES", "AMOUNT_OF_LEVELS_INJURED",
    "TYPE_A_INJURY", "TYPE_B_INJURY", "TYPE_C_INJURY", "TYPE_F_INJURY",
    # spinal region
    "C", "T", "L", "S",
    # mechanism
    "OORZAAKCATEGORIEID", "INTENTIE",
    # TBI
    "TBI_CT",
]
# Keep only cols present in df
schema_cols_present = [c for c in SCHEMA_COLS if c in df.columns]

miss = (df[schema_cols_present].isnull().mean() * 100).round(1).sort_values(ascending=False)
print(miss.to_string())

fig, ax = plt.subplots(figsize=(6, 10))
miss.plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(30, color="red", linestyle="--", linewidth=1, label="30% threshold")
ax.set_xlabel("% missing")
ax.set_title("Missingness per feature (included patients)")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/missingness_bar.png", dpi=150)
plt.show()

<a id="sec3b"></a>

### 3b. Missingness mechanism

Little's MCAR test across all partially observed numeric columns, then a closer look at `BMI` -
the only variable with substantial missingness - by clinical subgroup and by logistic regression.

Three terms are used below. **MCAR** (missing completely at random) means the chance of a value
being absent is unrelated to anything, observed or not. **MAR** (missing at random) means it
depends only on data we *do* observe, so the missing values can be predicted and imputed.
**MNAR** (missing not at random) means it depends on the unobserved value itself, which cannot
be fixed by imputation. Little's test can reject MCAR but cannot distinguish MAR from MNAR -
that distinction always rests on judgement.

If missingness is predictable from observed variables it is plausibly MAR rather than MCAR, which is what
justifies imputation instead of complete-case analysis.

In [ ]:
from scipy import stats as scipy_stats
import statsmodels.api as sm

# ── 1. Little's MCAR test ────────────────────────────────────────────────────
def little_mcar_test(data):
    """
    Little (1988) MCAR chi-square test.
    H0: data is MCAR. p < 0.05 suggests MAR or MNAR.
    Uses pairwise-complete covariance; patterns with n < 2 are skipped.
    """
    cols = [c for c in data.columns if 0 < data[c].isnull().mean() < 1.0]
    d = data[cols].copy()
    n, p = d.shape

    mu = d.mean()
    sigma = d.cov()

    miss = d.isnull()
    patterns = miss.apply(tuple, axis=1)

    d2, df_accum = 0.0, 0
    for pat, grp in miss.groupby(patterns.values):
        obs = [c for c, m in zip(cols, pat) if not m]
        if not obs or len(grp) < 2:
            continue
        ybar = d.loc[grp.index, obs].mean().values
        mu_j = mu[obs].values
        S_j  = sigma.loc[obs, obs].values
        try:
            S_inv = np.linalg.inv(S_j)
        except np.linalg.LinAlgError:
            S_inv = np.linalg.pinv(S_j)
        diff  = ybar - mu_j
        d2   += len(grp) * float(diff @ S_inv @ diff)
        df_accum += len(obs)

    df_free = max(df_accum - p, 1)
    p_val   = 1 - scipy_stats.chi2.cdf(d2, df_free)
    return d2, p_val, df_free


mcar_cols = [
    c for c in schema_cols_present
    if c in df.select_dtypes(include=[np.number]).columns
    and c not in ("ID", "DATUM_ONGEVAL", "DATUM_OVERLEDEN")
    and 0 < df[c].isnull().mean() < 1.0
]
d2, p_val, df_free = little_mcar_test(df[mcar_cols])
print(f"Little's MCAR test  (columns: {mcar_cols})")
print(f"  χ² = {d2:.2f},  df = {df_free},  p = {p_val:.4f}")
if p_val < 0.05:
    print("  → Reject MCAR (p < 0.05): missingness is likely MAR or MNAR.")
else:
    print("  → Cannot reject MCAR (p ≥ 0.05).")

# ── 2. Subgroup prevalence of BMI missingness ────────────────────────────────
print("\n\nBMI missingness rate by clinical subgroup")
print("=" * 50)
df["_bmi_miss"] = df["BMI"].isnull().astype(int)

subgroups = {
    "Age tertile": pd.qcut(df["LEEFTIJD_EXACT"], q=3, labels=["Young", "Middle", "Old"]),
    "Sex (0=M male, 1=V female)": df["Geslacht01"].astype("Int64"),
    "ISS severity": pd.cut(df["ISS08"], bins=[0, 14, 24, 76], labels=["<15", "15-24", "≥25"]),
    "MFI5 frailty (0 vs ≥1)": (df["MFI5"] >= 1).map({False: "Non-frail", True: "Frail"}),
    "D30_Mortality": df["D30_Mortality"].astype("Int64"),
}
for label, grouper in subgroups.items():
    tbl = (df.groupby(grouper, observed=True)["_bmi_miss"]
             .agg(n_miss="sum", n_total="count")
             .assign(pct=lambda x: (x["n_miss"] / x["n_total"] * 100).round(1)))
    print(f"\n{label}:")
    print(tbl.to_string())

# ── 3. Logistic regression predicting BMI missingness ────────────────────────
print("\n\nLogistic regression — predictors of BMI missingness")
print("=" * 50)

lr_features = [
    "LEEFTIJD_EXACT", "Geslacht01", "ISS08", "MFI5",
    "D30_Mortality", "NUMBER_OF_FRACTURES", "AMOUNT_OF_LEVELS_INJURED", "TBI_CT",
]
lr_df = df[lr_features + ["_bmi_miss"]].dropna()
print(f"Complete cases: {len(lr_df)} / {len(df)}")

X = sm.add_constant(lr_df[lr_features].astype(float))
y = lr_df["_bmi_miss"]
model = sm.Logit(y, X).fit(disp=0)

res_df = pd.DataFrame({
    "OR":     np.exp(model.params),
    "CI_low": np.exp(model.conf_int().iloc[:, 0]),
    "CI_high":np.exp(model.conf_int().iloc[:, 1]),
    "p":      model.pvalues,
}).drop("const").round(3).sort_values("p")

print(f"\nPseudo-R² (McFadden): {model.prsquared:.3f}")
print("\nOR (95% CI) for BMI missingness:")
print(res_df.to_string())

df.drop(columns=["_bmi_miss"], inplace=True)

<a id="sec4"></a>

## 4. Data integrity checks

Four checks:

1. **Identifiers** - is every patient uniquely identifiable, or are rows duplicated?
2. **Plausible ranges** - do age, BMI and injury severity fall within physically possible limits?
   An impossible value usually means a coding error rather than an unusual patient.
3. **Internal consistency** - do fields that should agree actually agree? A recorded date of
   death ought to imply that the mortality flag is set.
4. **Conformance to the data dictionary** - does each variable contain only the codes that the
   project schema (`data/schemas/patient.yaml`) says it may contain?

Columns still holding the registry's raw JA/NEE (yes/no) text at this point are skipped by check
4; they are converted to 1/0 in section 7.

In [ ]:
import yaml

print(f"Total included patients: {len(df)}")
# Count duplicates among rows that actually have an ID - a bare duplicated() would
# report the pair of missing IDs as a duplicate.
print(f"Missing IDs: {df['ID'].isna().sum()}")
print(f"Duplicate IDs (excluding missing): {df['ID'].dropna().duplicated().sum()}")

print()
print("=== range checks ===")
print(f"  Age  {df['LEEFTIJD_EXACT'].min():.1f} - {df['LEEFTIJD_EXACT'].max():.1f}"
      f"   <18: {(df['LEEFTIJD_EXACT'] < 18).sum()}   >120: {(df['LEEFTIJD_EXACT'] > 120).sum()}")
print(f"  BMI  {df['BMI'].min():.1f} - {df['BMI'].max():.1f}"
      f"   <10: {(df['BMI'] < 10).sum()}   >80: {(df['BMI'] > 80).sum()}")
print(f"  ISS  {df['ISS08'].min():.0f} - {df['ISS08'].max():.0f}"
      f"   >75: {(df['ISS08'] > 75).sum()}")

# A recorded date of death should imply Y1_MORTALITY = 1.
inconsistent = df["DATUM_OVERLEDEN"].notna() & (df["Y1_MORTALITY"] == 0.0)
print()

schema = yaml.safe_load(open("../data/schemas/patient.yaml", encoding="utf-8"))["columns"]
print()
print("=== values outside schema allowed_values ===")
checked, violations = 0, 0
for col, spec in schema.items():
    allowed = spec.get("allowed_values")
    if not allowed or col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
        continue
    checked += 1
    bad = df.loc[~df[col].isin(allowed) & df[col].notna(), col]
    if len(bad):
        violations += 1
        print(f"  {col:<24} {len(bad):>5} rows  values={sorted(bad.unique().tolist())}")
print(f"  none" if not violations else "")
print(f"  ({checked} numeric columns checked)")

<a id="sec5"></a>

## 5. Target distribution

In [ ]:
print("=== 30-day mortality ===")
print(df["D30_Mortality"].value_counts(dropna=False))
print(f"30-day mortality rate: {df['D30_Mortality'].mean():.1%}")

In [ ]:
# SPSS dates: seconds since 1582-10-14 (Gregorian calendar epoch).
# pd.Timestamp("1582-10-14") predates pandas' minimum Timestamp (~1677), causing overflow.
#
# Fix 1 — survival_days: subtract the two raw SPSS columns directly; epoch cancels out.
# Fix 2 — date_accident: offset by SPSS→Unix constant (1582-10-14 to 1970-01-01)
#          so pd.to_datetime receives a normal Unix timestamp.
SPSS_TO_UNIX_S = 12_219_292_800  # seconds from SPSS epoch to Unix epoch

df["survival_days"] = (df["DATUM_OVERLEDEN"] - df["DATUM_ONGEVAL"]) / 86400
df["date_accident"] = pd.to_datetime(df["DATUM_ONGEVAL"] - SPSS_TO_UNIX_S, unit="s")

print(f"Accident date range: {df['date_accident'].min():%Y-%m} — {df['date_accident'].max():%Y-%m}")

died = df[df["Y1_MORTALITY"] == 1.0]
print(f"Patients who died within 1 year: {len(died)}")
print(f"Survival days — median: {died['survival_days'].median():.0f}, "
      f"range: {died['survival_days'].min():.0f} — {died['survival_days'].max():.0f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(died["survival_days"].dropna(), bins=30, color="steelblue", edgecolor="white")
ax.axvline(30, color="red", linestyle="--", label="30-day mark")
ax.set_xlabel("Days from accident to death")
ax.set_ylabel("Count")
ax.set_title("Survival time distribution (patients who died within 1 year)")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/survival_time_dist.png", dpi=150)
plt.show()

<a id="sec6"></a>

## 6. Predictor distributions

Continuous variables first (check skewness — transform candidates).  
Then categorical/ordinal (check near-zero variance — flag for collapsing or dropping).  
Then INTENTIE with labels to guide grouping into clinical categories.

In [ ]:
cont_cols = ["LEEFTIJD_EXACT", "BMI", "ISS08", "NUMBER_OF_FRACTURES", "AMOUNT_OF_LEVELS_INJURED"]
fig, axes = plt.subplots(1, len(cont_cols), figsize=(16, 4))
for ax, col in zip(axes, cont_cols):
    data = df[col].dropna()
    ax.hist(data, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel(f"n={len(data)}, miss={df[col].isna().sum()}", fontsize=8)
plt.suptitle("Continuous predictor distributions")
plt.tight_layout()
plt.savefig("../outputs/continuous_distributions.png", dpi=150)
plt.show()

print(df[cont_cols].describe().round(1).to_string())

<a id="sec6b"></a>

### 6b. Normality tests — scaler decision

For each continuous predictor:
- **Skewness / kurtosis** — descriptive; kurtosis < 0 = platykurtic (flat distribution, no transform will help)
- **D'Agostino-Pearson** (`scipy.stats.normaltest`) — omnibus normality test; preferred over Shapiro-Wilk at n ≈ 1500
- **Yeo-Johnson optimal λ** — `PowerTransformer.fit()` estimates the best-fitting lambda; λ=1 is identity (already normal); |λ−1| > 0.3 means a power transform is warranted

**Decision:** apply `PowerTransformer(method='yeo-johnson', standardize=True)` to **all** continuous predictors.  
Rationale: when λ≈1 (near-normal data), Yeo-Johnson reduces to a near-linear transform and `standardize=True` gives the same zero-mean/unit-variance output as StandardScaler. Applying it uniformly is simpler and strictly more general — no separate StandardScaler step needed.  
Note: the pipeline model is penalised logistic regression, so scaling is required for the
ElasticNet penalty to apply comparably across coefficients.

> **What this section does not answer.** Skewness, D'Agostino-Pearson and the Yeo-Johnson
> lambda all describe the *marginal* distribution of a predictor. Logistic regression makes no
> distributional assumption about its predictors at all, so none of these tests says anything
> about the thing that does matter: whether the log-odds of death move linearly with the
> predictor. A perfectly normal predictor can have a U-shaped effect, and a badly skewed one can
> have a perfectly linear effect. Yeo-Johnson is applied here to put predictors on comparable
> scales for the ridge penalty, which is a real benefit, but it is **monotone** and therefore
> cannot repair a non-linear effect. That separate question is tested in
> [section 11c](#sec11c), which finds age non-linear and the other three adequately linear.


In [ ]:
from scipy import stats
from sklearn.preprocessing import PowerTransformer

cont_cols = ["LEEFTIJD_EXACT", "BMI", "ISS08", "NUMBER_OF_FRACTURES"]

rows = []
for col in cont_cols:
    data = df[col].dropna().values
    skewness = stats.skew(data)
    kurt = stats.kurtosis(data)
    _, da_p = stats.normaltest(data)

    pt = PowerTransformer(method="yeo-johnson")
    pt.fit(data.reshape(-1, 1))
    yj_lambda = pt.lambdas_[0]

    needs_transform = abs(yj_lambda - 1) > 0.3
    rows.append({
        "feature":          col,
        "n":                len(data),
        "skewness":         round(skewness, 3),
        "kurtosis":         round(kurt, 3),
        "dagostino_p":      f"{da_p:.2e}",
        "yj_lambda":        round(yj_lambda, 3),
        "|λ−1| > 0.3":     "yes" if needs_transform else "no",
    })

norm_df = pd.DataFrame(rows)
print(norm_df.to_string(index=False))
print()
print("Decision: PowerTransformer(yeo-johnson, standardize=True) applied to ALL continuous predictors.")
print("When λ≈1 it behaves like StandardScaler; standardize=True handles centering/scaling in both cases.")

In [ ]:
fig, axes = plt.subplots(2, len(cont_cols), figsize=(16, 8))

for j, col in enumerate(cont_cols):
    data = df[col].dropna().values

    stats.probplot(data, plot=axes[0, j])
    axes[0, j].set_title(f"{col}\n(original)", fontsize=9)

    pt = PowerTransformer(method="yeo-johnson")
    transformed = pt.fit_transform(data.reshape(-1, 1)).flatten()
    stats.probplot(transformed, plot=axes[1, j])
    axes[1, j].set_title(f"{col}\n(Yeo-Johnson λ={pt.lambdas_[0]:.2f})", fontsize=9)

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Yeo-Johnson", fontsize=10)
plt.suptitle("Q-Q plots: original vs Yeo-Johnson transformed continuous predictors", fontsize=11)
plt.tight_layout()
plt.savefig("../outputs/qq_plots_scaling.png", dpi=150)
plt.show()

In [ ]:
cat_cols = [
    "Geslacht01", "MFI5", "COMORB", "ASIA",
    "HT_CODE", "CHF_CODE", "COPD_CODE", "DM_CODE", "DEPENDENT_CODE", "N_CODE",
]
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts(dropna=False).sort_index()
    labels = [str(x) if pd.notna(x) else "NaN" for x in vc.index]
    ax.bar(labels, vc.values, color="steelblue")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel(f"miss={df[col].isna().sum()}", fontsize=8)
    ax.tick_params(axis="x", labelsize=7)
    for i, v in enumerate(vc.values):
        ax.text(i, v + 2, str(v), ha="center", fontsize=7)
for ax in axes[len(cat_cols):]:
    ax.set_visible(False)
plt.suptitle("Categorical/ordinal predictor distributions")
plt.tight_layout()
plt.savefig("../outputs/categorical_distributions.png", dpi=150)
plt.show()

<a id="sec6c"></a>

### 6c. Injury mechanism (`INTENTIE`) - raw codes before grouping

The raw SPSS codes and their value labels, so the clinical grouping applied in the next cell can be
checked against what was actually recorded. Labels come from `data/schemas/value_labels.json`.

In [ ]:
import json, pathlib

labels_path = pathlib.Path("../data/schemas/value_labels.json")
intentie_labels = json.loads(labels_path.read_text(encoding="utf-8")).get("INTENTIE", {})
label_map = {float(k): v for k, v in intentie_labels.items()}

print("Raw INTENTIE codes, SPSS labels and counts:")
for c, n in df["INTENTIE"].value_counts(dropna=False).sort_index().items():
    label = label_map.get(c, "?") if pd.notna(c) else "missing"
    print(f"  {str(c):>6}  {label:<58}  n={n}")

In [ ]:
# Group INTENTIE into 6 clinical categories.
# Must stay in sync with cleaning.intentie_map in config/pipeline.yaml.
intentie_map = {
    1.0:  "MVA",        # motorised vehicle
    2.0:  "MVA",        # motorcycle
    3.0:  "MVA",        # moped / scooter
    4.0:  "OVA",        # bicycle
    5.0:  "OVA",        # pedestrian
    6.0:  "OVA",        # other road
    7.0:  "violence",   # gunshot
    8.0:  "violence",   # stabbing
    9.0:  "violence",   # blunt assault
    10.0: "fall_low",   # low-energy fall (same level)
    11.0: "fall_high",  # high-energy fall (from height)
    12.0: "other",      # (absent from this cohort; kept to match config)
    13.0: "other",      # (absent from this cohort; kept to match config)
    14.0: "other",      # drowning
    15.0: "other",      # asphyxia
    18.0: "other",      # unspecified other
}
df["INTENTIE_cat"] = df["INTENTIE"].map(intentie_map)

vc = df["INTENTIE_cat"].value_counts(dropna=False)
print("INTENTIE_cat distribution:")
print(vc)
print()
print(f"Unmapped / missing: {df['INTENTIE_cat'].isnull().sum()}")

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
order = ["fall_low", "fall_high", "MVA", "OVA", "violence", "other"]
counts = df["INTENTIE_cat"].value_counts().reindex(order, fill_value=0)
ax.bar(order, counts.values, color="steelblue")
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha="center", fontsize=9)
ax.set_ylabel("Patients")
ax.set_title("Injury mechanism - grouped (INTENTIE_cat)")
plt.tight_layout()
plt.savefig("../outputs/intentie_cat.png", dpi=150)
plt.show()

<a id="sec6d"></a>

### 6d. `MFI5` - collapsing the sparse top grades

The distribution panel above shows `MFI5` is concentrated almost entirely at 0, with a long thin
tail. Entered as-is, grades 4 and 5 ask the model to estimate a level from fewer than 10 patients
between them. The table below shows what those levels actually contain.

In [ ]:
mfi_raw = (df.groupby("MFI5")["D30_Mortality"]
             .agg(n="size", deaths="sum", rate="mean"))
mfi_raw["mortality_%"] = (100 * mfi_raw["rate"]).round(1)

mfi_capped = (df.assign(MFI5_capped=df["MFI5"].clip(upper=3))
                .groupby("MFI5_capped")["D30_Mortality"]
                .agg(n="size", deaths="sum", rate="mean"))
mfi_capped["mortality_%"] = (100 * mfi_capped["rate"]).round(1)

print("MFI5 as recorded (0-5):")
print(mfi_raw[["n", "deaths", "mortality_%"]].to_string())
print("\nMFI5 capped at 3+:")
print(mfi_capped[["n", "deaths", "mortality_%"]].to_string())

print("\nMonotone in observed mortality?")
print(f"  as recorded : {mfi_raw['rate'].is_monotonic_increasing}")
print(f"  capped at 3 : {mfi_capped['rate'].is_monotonic_increasing}")

Grades 4 and 5 together hold fewer than 10 patients. Their observed mortality is not a signal
about frailty, it is what a handful of patients looks like. Those two points also break the
monotone gradient the rest of the scale shows, and because `MFI5` enters the model as a single
linear term (one coefficient across grades), a spurious value at the top grades pulls that slope.

Collapsing 3, 4 and 5 into a single **3+** grade puts 25 patients in the top level and restores
a clean monotone gradient: **12.2% -> 16.2% -> 19.2% -> 28.0%**. No information is lost that the
data could support in the first place - the distinction between 3, 4 and 5 deficits rests on 25
patients in total.

> **Decision:** `MFI5` is capped at 3 in `Cleaner`, via `cleaning.ordinal_caps` in
> `config/pipeline.yaml`. It stays a single ordinal term rather than becoming dummies: with
> 4 levels the dummy coding costs 2 extra parameters, and a likelihood-ratio test of dummies
> against the linear-capped term does not reach significance.

<a id="sec6e"></a>

### 6e. `NUMBER_OF_FRACTURES` - a count, capped at 6+

`config/pipeline.yaml` separates two treatments. Every column under `features.continuous`
receives a Yeo-Johnson transform and enters the model as one linear term; columns under
`features.ordinal` enter as one linear term without that transform. Neither list is a statement
about measurement scale, and logistic regression makes no assumption of continuity - the only
requirements are that the values be numeric and that the log-odds move linearly with them.

`NUMBER_OF_FRACTURES` is a count, not a continuous measurement. The questions are whether it has
enough distinct values, and a well-behaved enough shape, to be worth one coefficient rather than
a set of dummies, and whether its sparse tail needs a cap like the one applied to `MFI5` in 6d.

In [ ]:
frac = df["NUMBER_OF_FRACTURES"]
print(f"n = {frac.notna().sum()}   missing = {frac.isna().sum()}   "
      f"distinct values = {frac.nunique()}")
print(f"range {frac.min():.0f}-{frac.max():.0f}   median {frac.median():.0f}   "
      f"mean {frac.mean():.2f}   SD {frac.std():.2f}")
print(f"all values are whole numbers: {bool((frac.dropna() % 1 == 0).all())}")

counts = (pd.DataFrame({"k": frac, "y": df["D30_Mortality"]}).dropna()
            .groupby("k").agg(n=("y", "size"), deaths=("y", "sum"), rate=("y", "mean")))
counts["%_cohort"] = (100 * counts["n"] / counts["n"].sum()).round(1)
counts["cum_%"] = counts["%_cohort"].cumsum().round(1)
counts["mortality_%"] = (100 * counts["rate"]).round(1)
print("\nDistribution and 30-day mortality by fracture count:")
print(counts[["n", "%_cohort", "cum_%", "deaths", "mortality_%"]].to_string())

# How far out does the tail sit, before and after the transform the pipeline applies?
pt_frac = PowerTransformer("yeo-johnson")
frac_yj = pt_frac.fit_transform(frac.fillna(frac.median()).to_numpy()[:, None]).ravel()
raw = frac.fillna(frac.median()).to_numpy(dtype=float)
print(f"\nYeo-Johnson lambda = {pt_frac.lambdas_[0]:.3f}")
print("Leverage of the tail, in SD from the mean:")
for k in [8, 12, 18]:
    i = int(np.argmin(np.abs(raw - k)))
    print(f"  {k:>2d} fractures   raw {(k - raw.mean()) / raw.std():>5.2f} SD"
          f"   after Yeo-Johnson {frac_yj[i]:>5.2f} SD")

#### What the distribution shows

The count runs **0 to 18 across 17 distinct values**, median 2, mean 2.73, and is heavily
right-skewed: three quarters of patients have 3 or fewer fractures and 93% have 6 or fewer. The
upper tail is thin - counts of 9 or more cover 51 patients (3.4%), and each of the top four values
holds fewer than 5 patients.

**Seventeen distinct values is comfortably enough to spend one coefficient on.** Dummy-coding
would cost 16 parameters to model a relationship that section 11c shows is adequately linear
(LR chi2 = 1.63, p = 0.44 raw; 0.69, p = 0.71 transformed), and most of those dummies would rest
on a handful of patients. The linear term is the right call.

#### Does the sparse tail need capping, as `MFI5` did?

Yes. The tail is thin in the same way as the top `MFI5` grades: above 6 fractures, mortality
swings between extremes in counts held by only a few patients. Entered untransformed and
uncapped, the most extreme counts sit far from the mean (18 fractures is 6.4 SD out) and could
set the slope by themselves.

Two remedies were compared. A Yeo-Johnson transform compresses the tail; at lambda = -0.266 it
is close to a reciprocal:

| Fractures | Raw, SD from mean | After Yeo-Johnson |
|---|---|---|
| 8 | 2.21 | 1.78 |
| 12 | 3.89 | 2.28 |
| 18 | 6.41 | 2.75 |

Capping at 6 instead pools the 156 patients (10.5%) with 6 or more fractures into one top level,
which then sits 2.04 SD from the mean - closer in than the transform brings the extreme.

Fit does not separate the codings that were tested:

| Specification | AIC | beta | p |
|---|---|---|---|
| raw linear, 0-18 | 698.7 | +0.030 | 0.83 |
| Yeo-Johnson linear | 698.1 | +0.131 | 0.43 |
| capped at 8, then Yeo-Johnson | 698.0 | +0.139 | 0.41 |
| capped at 6, then Yeo-Johnson | 697.7 | +0.164 | 0.32 |
| log(1 + k) | 698.2 | +0.111 | 0.49 |

Every option sits within 0.4 AIC of every other, and the predictor is weak however it is coded -
the adjusted coefficient never approaches significance (p = 0.32 to 0.83), most likely because
`ISS08` already carries overall injury burden. The final coding, capped at 6 without a transform,
was not part of this comparison.

With fit unable to choose, the coding is chosen for interpretation and robustness. The count is
capped at 6 and entered **untransformed**: exp(beta) is then an odds ratio per additional
fracture rather than per SD of a transformed count, the cap limits the tail's leverage more than
the transform does, and listing the predictor as ordinal keeps imputed values whole numbers.

> **Decision:** `NUMBER_OF_FRACTURES` is capped at 6+ (`cleaning.ordinal_caps` in
> `config/pipeline.yaml`) and listed under `features.ordinal`, so it skips the Yeo-Johnson
> transform and enters as a single linear term. No dummies.

#### Data-dictionary correction

56 patients (3.8%) have **0** fractures - patients seen for a spinal consultation where no
fracture was ultimately classified. `data/schemas/patient.yaml` declared `min: 1.0` for this
column, which those rows violate. The schema is documentation only and is not enforced anywhere
in the pipeline, so nothing downstream was affected, but the bound has been corrected to `0.0`.

<a id="sec7"></a>

## 7. Spinal region involvement

Prevalence and 30-day mortality for each of the four CTLS regions. The regions are not mutually
exclusive - a patient can have injuries at several levels - so they are kept as four binaries.

In [ ]:
region_cols   = ["C", "T", "L", "S"]
region_labels = {"C": "Cervical", "T": "Thoracic", "L": "Lumbar", "S": "Sacral"}

# Normalize JA/NEE → 1/0 so all comparisons work regardless of encoding
for col in region_cols:
    df[col] = df[col].map({"JA": 1, "NEE": 0, 1: 1, 1.0: 1, 0: 0, 0.0: 0})

print("=== Regional involvement prevalence ===")
for col in region_cols:
    n = df[col].eq(1).sum()
    print(f"  {region_labels[col]}: {n} ({n/len(df):.1%})")

df["n_regions"] = df[region_cols].eq(1).sum(axis=1)
print("\n=== Number of regions involved per patient ===")
print(df["n_regions"].value_counts().sort_index().to_string())

print("\n=== 30-day mortality by regional involvement ===")
for col in region_cols:
    mort_yes = df.loc[df[col] == 1, "D30_Mortality"].mean()
    mort_no  = df.loc[df[col] == 0, "D30_Mortality"].mean()
    print(f"  {region_labels[col]}: present={mort_yes:.1%}  absent={mort_no:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

prev = df[region_cols].eq(1).mean() * 100
axes[0].bar([region_labels[c] for c in region_cols], prev.values, color="steelblue")
axes[0].set_ylabel("% patients")
axes[0].set_title("Regional spinal involvement — prevalence")
for i, v in enumerate(prev.values):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)

mort_vals = [df.loc[df[c] == 1, "D30_Mortality"].mean() * 100 for c in region_cols]
axes[1].bar([region_labels[c] for c in region_cols], mort_vals, color="steelblue")
axes[1].set_ylabel("30-day mortality (%)")
axes[1].set_title("30-day mortality by spinal region involved")
for i, v in enumerate(mort_vals):
    axes[1].text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("../outputs/spinal_region_analysis.png", dpi=150)
plt.show()

<a id="sec7b"></a>

### 7b. Number of CTLS regions involved (`n_regions`)

`n_regions` = 0-4, derived from the C/T/L/S flags above. Explored here but dropped from the model:
it is collinear with `NUMBER_OF_FRACTURES` (section 9).

In [ ]:
# n_regions = 0–4, computed above from C/T/L/S binary flags

print("=== n_regions distribution ===")
print(df["n_regions"].value_counts().sort_index().to_string())
print(f"\nMissing: {df['n_regions'].isna().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Distribution
vc = df["n_regions"].value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color="steelblue")
axes[0].set_xlabel("Number of CTLS regions involved")
axes[0].set_ylabel("Patients")
axes[0].set_title("Distribution of n_regions")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 2, str(v), ha="center", fontsize=9)

# 30-day mortality by n_regions
mort_by_n = df.groupby("n_regions")["D30_Mortality"].agg(["mean", "count"])
axes[1].bar(mort_by_n.index.astype(str), mort_by_n["mean"] * 100, color="steelblue")
axes[1].set_xlabel("Number of CTLS regions involved")
axes[1].set_ylabel("30-day mortality (%)")
axes[1].set_title("30-day mortality by n_regions")
for i, (idx, row) in enumerate(mort_by_n.iterrows()):
    axes[1].text(i, row["mean"] * 100 + 0.3, f"{row['mean']:.1%}\n(n={int(row['count'])})",
                 ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/n_regions_analysis.png", dpi=150)
plt.show()

<a id="sec8"></a>

## 8. Neurological status: `N_CODE`, `ASIA`, and the not-examinable state

`N_CODE` (AO Spine neurological modifier) and `ASIA` both describe neurological status, so they
must be audited together *before* the collinearity screen - running that screen on the raw codes
gives a misleading answer, as shown below.

The frequencies and cross-tabulations in the next cell establish three things:

1. what the undocumented `N_CODE` level `0` actually is,
2. that `NX` and an unrecorded `ASIA` are the same patients, and
3. how strongly that state predicts death.

In [ ]:
# Accident year is needed here and reused by the enrollment section below.
df["year"] = df["date_accident"].dt.year

n_lab = {0: "0 (undocumented)", 1: "N0 intact", 2: "N1 transient", 3: "N2 radicular",
         4: "N3 incomplete", 5: "N4 complete", 6: "NX not examinable"}

print("=== N_CODE frequency and 30-day mortality ===")
print(df.groupby(df["N_CODE"].astype("Int64").map(n_lab), sort=False)["D30_Mortality"]
        .agg(n="size", died="sum", mortality="mean")
        .sort_values("n", ascending=False)
        .to_string(float_format=lambda v: f"{v:.3f}"))

# (1) Is level 0 a severity level, or simply not collected?
print()
print("=== share of patients with N_CODE == 0, by accident year ===")
print(df.groupby("year")["N_CODE"].apply(lambda s: (s == 0).mean()).round(2).to_string())

# (2) Do NX and an unrecorded ASIA describe the same patients?
#     Sentinel 888 already became NaN in the loading cell; ASIA 0 = 'not determinable'.
asia_unrec = df["ASIA"].isna() | df["ASIA"].eq(0)
print()
print("=== ASIA grade by N_CODE level (unrec. = 0 or missing) ===")
print(pd.crosstab(df["N_CODE"].astype("Int64").map(n_lab),
                  df["ASIA"].fillna(-1).astype(int).replace({-1: "unrec.", 0: "unrec."}),
                  dropna=False).to_string())

# (3) How prognostic is the not-examinable state?
print()
print("=== 30-day mortality by whether ASIA was recorded ===")
print(df.groupby(asia_unrec.map({False: "ASIA recorded", True: "ASIA unrecorded"}))["D30_Mortality"]
        .agg(n="size", died="sum", mortality="mean")
        .to_string(float_format=lambda v: f"{v:.3f}"))

### What the tables show

**`N_CODE` level `0` means 'not recorded', not a severity level.** It appears in 100% of 2019 and
2020 records and in 0% of every other year: the AO N-modifier was not collected in that period.
Those patients have fully recorded ASIA grades spread across 1-5, and their 30-day mortality
(13.0%) is exactly the cohort base rate. Left as-is it enters the model as the numeric level
*below* N0-intact.

**An unrecorded `ASIA` always means `NX`, though not every `NX` patient has one.** Every patient with an unrecorded ASIA is
`NX`, and no patient outside `NX` has one. `NX` does not mean 'worse than N4 complete' - it means
the neurological exam could not be performed (intubated, sedated, severe head injury). Of the
146 `NX` patients, 118 have no ASIA grade and 28 do - and those 28 have a 30-day mortality of
10.7%, close to the cohort average, against 80.5% for the 118. The prognostic signal therefore
follows the *missing ASIA grade*, not the `NX` label, which is why the indicator below is
defined from `ASIA` rather than from `N_CODE`.

**That state is the strongest single signal in the cohort:** 30-day mortality of 80.5% versus 7.5%.
Treating it as ordinary missingness hands it to the imputer, which fills a plausible ASIA grade and
destroys it. This is missing-not-at-random, and it needs an explicit indicator.

### Recode

| New variable | Definition | Purpose |
|---|---|---|
| `NEURO_NOT_EXAMINABLE` | 1 if ASIA unrecorded (0 or missing) | carries the not-examinable signal explicitly |
| `ASIA_exam` | ASIA, set to NaN where not examinable | severity among patients who *could* be examined |
| `N_CODE_valid` | N_CODE restricted to N0-N4 (`0` and `NX` set to NaN) | severity on its documented scale only |

`ASIA_exam` and `N_CODE_valid` exist for the collinearity screen in section 9. The pipeline itself
keeps `ASIA` (pinned to grade 5 where not examinable, so the indicator carries the signal) and
drops `N_CODE` entirely - see `config/pipeline.yaml` and `Cleaner`.

In [ ]:
# Mirrors Cleaner (src/preprocessing/transformers.py).
df["NEURO_NOT_EXAMINABLE"] = asia_unrec.astype(int)

# Analysis-only views: each variable restricted to the rows where it is meaningful.
df["ASIA_exam"] = df["ASIA"].where(~asia_unrec)
df["N_CODE_valid"] = df["N_CODE"].where(df["N_CODE"].between(1, 5))

print(f"NEURO_NOT_EXAMINABLE = 1 : {int(df['NEURO_NOT_EXAMINABLE'].sum())}")
print(f"ASIA_exam non-missing    : {int(df['ASIA_exam'].notna().sum())}")
print(f"N_CODE_valid non-missing : {int(df['N_CODE_valid'].notna().sum())}"
      f"  (excludes {int(df['N_CODE'].eq(0).sum())} not-recorded and {int(df['N_CODE'].eq(6).sum())} NX)")

rho_raw = df["N_CODE"].corr(df["ASIA"], method="spearman")
rho_fix = df["N_CODE_valid"].corr(df["ASIA_exam"], method="spearman")
print()
print(f"Spearman N_CODE vs ASIA, raw codes      : {rho_raw:.3f}")
print(f"Spearman N_CODE_valid vs ASIA_exam      : {rho_fix:.3f}")
print("-> the raw-code value is diluted by the 0 and NX levels and would not trip a 0.6 screen.")

<a id="sec9"></a>

## 9. Collinearity

Spearman correlation across all predictors. Sentinels (888/999) were set to NaN in the loading
cell, so no separate `ASIA_clean` column is needed. Pairs with |r| > 0.6 are flagged for
redundancy review.

This screen uses the **recoded** neurological variables from section 8 - `ASIA_exam`,
`N_CODE_valid` and `NEURO_NOT_EXAMINABLE` - rather than the raw codes.

Correlations are computed pairwise-complete, so pairs involving `ASIA_exam` or `N_CODE_valid`
use fewer rows than the full cohort; the n for the key pair is printed below the matrix.
`NEURO_NOT_EXAMINABLE` is constant within the rows where `ASIA_exam` and `N_CODE_valid` exist,
so those two correlations are undefined (blank in the matrix) - by construction, since the
indicator marks exactly the rows the other two exclude.

In [ ]:
pred_cols = [c for c in [
    "LEEFTIJD_EXACT", "Geslacht01", "BMI",
    "MFI5", "HT_CODE", "CHF_CODE", "COPD_CODE", "DM_CODE", "DEPENDENT_CODE", "COMORB",
    "ISS08", "ASIA_exam", "SCI_yes_no", "N_CODE_valid", "NEURO_NOT_EXAMINABLE",
    "NUMBER_OF_FRACTURES", "AMOUNT_OF_LEVELS_INJURED", "n_regions",
    "TYPE_A_INJURY", "TYPE_B_INJURY", "TYPE_C_INJURY", "TYPE_F_INJURY",
    "TBI_CT",
] if c in df.columns]

corr = df[pred_cols].astype(float).corr(method="spearman")

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    ax=ax, annot_kws={"size": 7}, linewidths=0.3,
)
ax.set_title("Spearman correlation - predictors (recoded neurological variables)")
plt.tight_layout()
plt.savefig("../outputs/correlation_matrix.png", dpi=150)
plt.show()

pairs = [
    (corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 3))
    for i in range(len(corr.columns))
    for j in range(i + 1, len(corr.columns))
    if abs(corr.iloc[i, j]) > 0.6
]
if pairs:
    print("Pairs with |Spearman r| > 0.6:")
    for a, b, r in sorted(pairs, key=lambda x: -abs(x[2])):
        n_pair = df[[a, b]].dropna().shape[0]
        print(f"  {a:<26} vs {b:<26}  r={r:>7}  (n={n_pair})")
else:
    print("No pairs with |Spearman r| > 0.6")

print()

<a id="sec9b"></a>

### 9b. Collinearity within the final predictor set

The matrix above is a *screening* matrix: it deliberately contains variables that were later
dropped. For three of them it is also the evidence for the drop - `SCI_yes_no`, `N_CODE` and
`AMOUNT_OF_LEVELS_INJURED` are each flagged above 0.6. For four others it is not.
`CHF_CODE`, `COPD_CODE`, `DM_CODE` and `DEPENDENT_CODE` sit well below the threshold and appear
in it unflagged: they are dropped because each is an *item* of `MFI5`, which is retained in their
place. That is a redundancy the matrix cannot show, since a sum need not correlate strongly with
any one of its components (the largest here is `MFI5` vs `HT_CODE` at 0.81; the rest are lower).
See 2b.

One predictor is missing from the matrix entirely: `INTENTIE_cat`. It is nominal, so it has no
rank and cannot enter a Spearman matrix - that is why it is absent from both matrices, not a
judgement that it needed no check.

This cell repeats the collinearity check on the final candidate predictors only - the set that
goes into the model - and then screens `INTENTIE_cat` against every one of them separately. For
the continuous predictors that is the correlation ratio eta (the square root of the share of
their variance lying between mechanism groups); for the binary and ordinal ones it is Cramer's V.
Both are on the same 0-1 scale as |r|, so the 0.6 threshold used above carries over unchanged.
`ASIA` is shown as `ASIA_exam` (restricted to examinable patients) so that the not-examinable
rows do not distort it, exactly as in 9 above.


In [ ]:
final_cols = [
    "LEEFTIJD_EXACT", "BMI", "ISS08", "NUMBER_OF_FRACTURES",
    "Geslacht01", "TBI_CT", "TYPE_A_INJURY", "TYPE_B_INJURY", "TYPE_C_INJURY",
    "TYPE_F_INJURY", "C", "T", "L", "S", "NEURO_NOT_EXAMINABLE",
    "COMORB", "ASIA_exam", "MFI5",
]

# C/T/L/S were converted to 1/0 in section 7, so all of these are numeric by now.
final_corr = df[final_cols].astype(float).corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(final_corr, dtype=bool))
sns.heatmap(final_corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7}, linewidths=0.3)
ax.set_title("Spearman correlation - final candidate predictors")
plt.tight_layout()
plt.savefig("../outputs/correlation_matrix_final.png", dpi=150)
plt.show()

pairs = [
    (final_corr.columns[i], final_corr.columns[j], round(final_corr.iloc[i, j], 3))
    for i in range(len(final_corr.columns))
    for j in range(i + 1, len(final_corr.columns))
    if abs(final_corr.iloc[i, j]) > 0.6
]
# INTENTIE_cat is one feature but 6 model parameters: 7 levels (the 6 of section 6c plus
# the explicit `unknown` of cleaning.intentie_unknown) minus the fall_low reference.
print(f"Numeric predictors checked here: {len(final_cols)}")
print(f"Plus INTENTIE_cat as 6 dummies -> {len(final_cols) + 6} model parameters "
      f"across {len(final_cols) + 1} features")
print(f"Largest |Spearman r| in the set: "
      f"{max(abs(final_corr.values[~np.eye(len(final_cols), dtype=bool)])):.3f}")
if pairs:
    print()
    print("Pairs above 0.6:")
    for a, b, r in sorted(pairs, key=lambda x: -abs(x[2])):
        print(f"  {a:<24} vs {b:<24} r={r}")
else:
    print()
    print("No pair exceeds 0.6 - no redundancy among the numeric predictors.")

# --- INTENTIE_cat: nominal, so screened outside the matrix -------------------------
from scipy import stats


def correlation_ratio(values, groups):
    """Eta for a numeric variable across nominal groups: sqrt(between-SS / total-SS)."""
    grand = values.mean()
    between = sum(len(v) * (v.mean() - grand) ** 2 for _, v in values.groupby(groups))
    return np.sqrt(between / ((values - grand) ** 2).sum())


def cramers_v(a, b):
    """Cramer's V for two nominal variables, on the same 0-1 scale as |r|."""
    table = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(table)[0]
    return np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))


# Blank and unmapped codes become "unknown", matching cleaning.intentie_unknown in the config.
mech = df["INTENTIE_cat"].fillna("unknown")
eta_cols = ["LEEFTIJD_EXACT", "BMI", "ISS08", "NUMBER_OF_FRACTURES"]

print()
print("INTENTIE_cat vs each final predictor (eta for continuous, Cramer's V otherwise):")
assoc = []
for col in final_cols:
    sub = df[[col]].assign(mech=mech).dropna()
    if col in eta_cols:
        stat = correlation_ratio(sub[col], sub["mech"])
        groups = [v.values for _, v in sub[col].groupby(sub["mech"]) if len(v) > 1]
        _, p = stats.kruskal(*groups)
        detail = f"eta  (Kruskal-Wallis p={p:.1e})"
    else:
        stat = cramers_v(sub["mech"], sub[col])
        detail = "V"
    assoc.append((col, stat))
    flag = "   <- above 0.6" if stat > 0.6 else ""
    print(f"  {col:<24} {stat:.3f}  {detail:<34} (n={len(sub)}){flag}")

worst_col, worst = max(assoc, key=lambda x: x[1])
print()
print(f"Strongest association with mechanism: {worst_col} ({worst:.3f})")
if worst > 0.6:
    print("Above 0.6 - review before keeping both.")
else:
    print("Below 0.6 - INTENTIE_cat is not redundant with any predictor in the set.")


Nothing in the set reaches 0.6, so `INTENTIE_cat` is kept and no predictor is dropped on the
strength of this screen. Two of the numbers are worth recording.

`ISS08` - the association this check was added to settle - is real but modest: eta = 0.219
(Kruskal-Wallis p = 1.6e-17), against median `ISS08` of 16 for same-level falls and 22 for falls
from height (section 2e). Mechanism accounts for roughly 5% of the variance in `ISS08`
(eta^2 = 0.048). Clearly non-zero, nowhere near enough for one to stand in for the other.

The stronger link is with **age**: eta = 0.500, which is the largest association anywhere in the
final set - marginally above the 0.495 maximum of the Spearman matrix itself, and on a comparable
0-1 scale. Same-level falls are overwhelmingly an elderly presentation while falls from height,
vehicle accidents and violence are younger, so mechanism and age carry partly the same
information. Both are kept, since 0.500 is well inside the threshold, but this is the pair to
watch: age enters as a spline (section 11c) and mechanism as 6 dummies, which is 8 of the 24
model parameters between them. If either set turns out unstable in the bootstrap (phase 4), this
overlap is the first thing to check.


<a id="sec10"></a>

## 10. Enrollment over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

year_counts = df.groupby("year").size()
axes[0].bar(year_counts.index.astype(str), year_counts.values, color="steelblue")
axes[0].set_xlabel("Year of accident")
axes[0].set_ylabel("Patients enrolled")
axes[0].set_title("Enrollment over time")
for i, v in enumerate(year_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontsize=9)

mort_by_year = df.groupby("year")["D30_Mortality"].agg(["mean", "count"])
axes[1].bar(mort_by_year.index.astype(str), mort_by_year["mean"] * 100, color="steelblue")
axes[1].set_xlabel("Year of accident")
axes[1].set_ylabel("30-day mortality (%)")
axes[1].set_title("30-day mortality rate by year (drift check)")
for i, (_, row) in enumerate(mort_by_year.iterrows()):
    axes[1].text(i, row["mean"] * 100 + 0.3, f"{row['mean']:.1%}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/enrollment_over_time.png", dpi=150)
plt.show()

print("30-day mortality by year:")
print(mort_by_year.rename(columns={"mean": "rate", "count": "n"})
      .assign(rate=lambda x: x["rate"].round(3)).to_string())

<a id="sec11"></a>

## 11. Pilot multivariate logistic regression

Purposes:
1. Estimate how much signal the candidate predictors carry, and from that how many predictors this
   sample can support without excessive overfitting
2. Give a first multivariate read on the candidate predictor set

The predictor set mirrors the pipeline's candidate features in `config/pipeline.yaml`: it includes
`MFI5` and
`NEURO_NOT_EXAMINABLE` and excludes `N_CODE` (see section 8). `INTENTIE_cat` is one-hot encoded
with `fall_low` as reference, giving 5 dummies.

**What R2_cs is.** Cox-Snell R-squared, a likelihood-based measure of explained variation that
extends R-squared to logistic models. Riley and colleagues showed that the sample size a
prediction model needs depends on it: the more signal the predictors carry, the more parameters
a given sample can support. Their shrinkage criterion - criterion (i) in *Stat Med*
2019;38:1276-96, relabelled **B3** in *BMJ* 2020;368:m441 - asks for a sample large enough that
coefficients need shrinking by no more than 10% (uniform shrinkage factor S >= 0.9).
Rearranging it gives the largest defensible number of parameters for this dataset. The full
assessment against all four criteria is in the sample-size notebook.

Missing values are filled by simple median/mode imputation in the next cell. That is a pilot
shortcut only - the pipeline uses `IterativeImputer` fitted inside each CV fold.

In [ ]:
# Simple median/mode imputation for pilot only - NOT for the final pipeline.
# Purpose: recover complete cases lost to listwise deletion (primarily BMI).
from sklearn.impute import SimpleImputer

# NEURO_NOT_EXAMINABLE was derived in section 8; ASIA is pinned below so the
# not-examinable rows never reach the imputer with a fabricated grade.

pilot_all_cols = [
    "LEEFTIJD_EXACT", "Geslacht01", "BMI",
    "COMORB", "MFI5",
    "ISS08", "ASIA", "NUMBER_OF_FRACTURES", "NEURO_NOT_EXAMINABLE",
    "TYPE_A_INJURY", "TYPE_B_INJURY", "TYPE_C_INJURY", "TYPE_F_INJURY",
    "TBI_CT", "C", "T", "L", "S", "INTENTIE_cat", "D30_Mortality",
]

continuous_cols = ["LEEFTIJD_EXACT", "BMI", "ISS08", "NUMBER_OF_FRACTURES"]
other_cols = [c for c in pilot_all_cols if c not in continuous_cols + ["INTENTIE_cat", "D30_Mortality"]]

df_imp = df[pilot_all_cols].copy()

# Pin ASIA for the not-examinable group; the indicator carries the signal.
df_imp.loc[df_imp["NEURO_NOT_EXAMINABLE"] == 1, "ASIA"] = 5

# Median imputation for continuous
imp_median = SimpleImputer(strategy="median")
df_imp[continuous_cols] = imp_median.fit_transform(df_imp[continuous_cols])

# Mode imputation for binary/ordinal/categorical
imp_mode = SimpleImputer(strategy="most_frequent")
df_imp[other_cols] = imp_mode.fit_transform(df_imp[other_cols])

# INTENTIE_cat: fill remaining NaN with mode
df_imp["INTENTIE_cat"] = df_imp["INTENTIE_cat"].fillna(df["INTENTIE_cat"].mode()[0])

print(f"Rows before imputation: {df[pilot_all_cols].dropna().shape[0]}")
print(f"Rows after imputation:  {df_imp.dropna().shape[0]}")
print(f"Events after imputation: {int(df_imp['D30_Mortality'].sum())}")
print(f"NEURO_NOT_EXAMINABLE = 1: {int(df_imp['NEURO_NOT_EXAMINABLE'].sum())}")
print(f"Remaining missing: {df_imp.isnull().sum().sum()}")

In [ ]:
import statsmodels.api as sm
from scipy.stats import chi2

# --- Build feature matrix ---
pilot_cols = [
    "LEEFTIJD_EXACT", "Geslacht01", "BMI",
    "COMORB", "MFI5",
    "ISS08", "ASIA", "NUMBER_OF_FRACTURES", "NEURO_NOT_EXAMINABLE",
    "TYPE_A_INJURY", "TYPE_B_INJURY", "TYPE_C_INJURY", "TYPE_F_INJURY",
    "TBI_CT", "C", "T", "L", "S",
]

# Use median/mode-imputed dataframe from cell above
intentie_dummies = pd.get_dummies(
    df_imp["INTENTIE_cat"], prefix="INTENTIE", drop_first=False
).astype(int)
intentie_dummy_cols = [c for c in intentie_dummies.columns if c != "INTENTIE_fall_low"]

df_pilot = df_imp[pilot_cols + ["D30_Mortality"]].copy()
df_pilot = df_pilot.join(intentie_dummies[intentie_dummy_cols])
df_pilot = df_pilot.dropna()

feature_cols = pilot_cols + intentie_dummy_cols
X = sm.add_constant(df_pilot[feature_cols].astype(float))
y = df_pilot["D30_Mortality"].astype(float)

pilot_model = sm.Logit(y, X).fit(disp=0)

# --- R²_cs and R²_Nagelkerke ---
n = len(y)
log_lik_null = pilot_model.llnull
r2_cs = 1 - np.exp((2 / n) * (log_lik_null - pilot_model.llf))
max_r2_cs = 1 - np.exp((2 / n) * log_lik_null)
r2_nag = r2_cs / max_r2_cs

print(f"N (after imputation): {n}   Events: {int(y.sum())}")
print(f"Log-likelihood (null): {log_lik_null:.2f}   (model): {pilot_model.llf:.2f}")
print(f"R²_cs            = {r2_cs:.4f}")
print(f"R²_Nagelkerke    = {r2_nag:.4f}")
print(f"max(R²_cs)       = {max_r2_cs:.4f}")
print(f"AIC              = {pilot_model.aic:.1f}")

# --- Riley's shrinkage criterion: the largest P for which expected uniform shrinkage stays
#     at or above S = 0.9, i.e. no more than 10% overfitting, at the observed R2_cs ---
def n_shrinkage(P, r2_cs, S=0.9):
    return P / ((S - 1) * np.log(1 - r2_cs / S))

max_p_safe = max(p for p in range(1, 200) if n_shrinkage(p, r2_cs) <= n)
print(f"Max P at <=10% expected shrinkage, observed R2_cs={r2_cs:.3f}: {max_p_safe}")
print(f"Events per parameter at P={len(feature_cols)}: {y.sum() / len(feature_cols):.1f}")
print(f"Current candidate P: {len(feature_cols)} — adequate: {len(feature_cols) <= max_p_safe}")

# --- Coefficients table ---
summary = pd.DataFrame({
    "OR":     np.exp(pilot_model.params),
    "CI_low": np.exp(pilot_model.conf_int()[0]),
    "CI_high":np.exp(pilot_model.conf_int()[1]),
    "p":      pilot_model.pvalues,
}).drop(index="const").round({"OR": 2, "CI_low": 2, "CI_high": 2, "p": 4})
summary["sig"] = summary["p"].apply(lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "")))
print("Multivariate ORs (median/mode-imputed pilot):")
print(summary.sort_values("p").to_string())

<a id="sec11b"></a>

### 11b. Cross-validated R²_cs — go/no-go check

The apparent R²_cs above is optimistic because the model is evaluated on the same data it was trained on. For the Riley B3 go/no-go decision (is our sample adequate for P candidate predictors?), we need the **expected new-data R²_cs**, not the in-sample value.

5-fold cross-validated predicted probabilities give a less biased estimate. Two caveats on the number below. The model is ridge-penalised (scikit-learn's `LogisticRegression` applies an L2 penalty by default). And the median/mode imputation was fitted on the whole dataset *before* cross-validation, so a little optimism leaks into the estimate; since BMI missingness is strongly outcome-associated (section 3b), that leak is not negligible. Both make this an approximation of expected new-data performance rather than a clean estimate of it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_cv = df_pilot[feature_cols].astype(float).values
y_cv = df_pilot["D30_Mortality"].astype(float).values
n_cv = len(y_cv)
phi_cv = y_cv.mean()

# 5-fold CV predicted probabilities — scale inside the pipeline to avoid data leakage
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, solver="lbfgs")),
])
probs_cv = cross_val_predict(pipe, X_cv, y_cv, cv=5, method="predict_proba")[:, 1]

# Clip to avoid log(0)
probs_cv = np.clip(probs_cv, 1e-10, 1 - 1e-10)

# Log-likelihoods
ll_null = np.sum(y_cv * np.log(phi_cv) + (1 - y_cv) * np.log(1 - phi_cv))
ll_cv   = np.sum(y_cv * np.log(probs_cv) + (1 - y_cv) * np.log(1 - probs_cv))

r2_cs_cv  = 1 - np.exp((2 / n_cv) * (ll_null - ll_cv))
r2_nag_cv = r2_cs_cv / max_r2_cs

optimism  = r2_cs - r2_cs_cv  # apparent minus CV

max_p_cv = max(p for p in range(1, 200) if n_shrinkage(p, r2_cs_cv) <= n_cv)

print(f"Apparent  R²_cs = {r2_cs:.4f}   R²_Nagelkerke = {r2_nag:.4f}")
print(f"CV (5-fold) R²_cs = {r2_cs_cv:.4f}   R²_Nagelkerke = {r2_nag_cv:.4f}")
print(f"Optimism (apparent − CV) = {optimism:.4f}")
print()
print(f"Max P at <=10% expected shrinkage, CV R2_cs = {r2_cs_cv:.3f}: {max_p_cv}")
print(f"Current candidate P: {len(feature_cols)} — adequate: {len(feature_cols) <= max_p_cv}")

<a id="sec11c"></a>

### 11c. Functional form - is a linear term adequate for the continuous predictors?

Sections 11 and 11b entered every continuous predictor as a single linear term on the logit
scale. The pipeline does the same: `Transformer` applies a Yeo-Johnson power transform and
`Scaler` standardises, but both are **monotone**, so the model still assumes the log-odds of
death move in one direction across the whole range of each predictor. If the true relationship
turns - as age-related mortality often does, being flat through middle age and steep in the
elderly - no monotone transform can represent it, and the fitted coefficient is a compromise
slope that fits neither end. Section 6b tested whether these predictors are *normally
distributed*, which is a different question and cannot detect this.

Each of the four continuous predictors is therefore compared, linear term against a **restricted
cubic spline** with 4 knots at the 5th, 35th, 65th and 95th percentiles (Harrell's default
placement). The spline basis is built so that its first column *is* the linear term, so dropping
the remaining columns recovers the linear model exactly and twice the difference in
log-likelihood is a likelihood-ratio test for linearity on 2 degrees of freedom.

Each model is otherwise the full pilot model of section 11, so the test asks whether the
non-linearity survives adjustment for the other predictors. Both the raw scale and the
Yeo-Johnson scale the pipeline actually fits on are tested, because the question is not only
"is this predictor non-linear" but "does the transform already in the pipeline handle it".

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.preprocessing import PowerTransformer

CONTINUOUS = ["LEEFTIJD_EXACT", "BMI", "ISS08", "NUMBER_OF_FRACTURES"]
KNOT_QUANTILES = {
    3: [0.10, 0.50, 0.90],
    4: [0.05, 0.35, 0.65, 0.95],
    5: [0.05, 0.275, 0.50, 0.725, 0.95],
}
y_ff = df_pilot["D30_Mortality"].astype(float).to_numpy()


def rcs_basis(x, knots):
    """Harrell restricted cubic spline basis: k knots -> k-1 columns, standardised.

    Column 0 is the linear term, so a model on the full basis nests the linear
    model and the log-likelihood difference is a test for linearity.
    """
    x = np.asarray(x, dtype=float)
    k = np.asarray(knots, dtype=float)
    K = len(k)
    den = (k[-1] - k[0]) ** 2
    cube = lambda v: np.where(v > 0, v ** 3, 0.0)
    cols = [x]
    for j in range(K - 2):
        cols.append(
            (cube(x - k[j])
             - cube(x - k[K - 2]) * (k[K - 1] - k[j]) / (k[K - 1] - k[K - 2])
             + cube(x - k[K - 1]) * (k[K - 2] - k[j]) / (k[K - 1] - k[K - 2])) / den
        )
    basis = np.column_stack(cols)
    # Standardised purely for numerical conditioning: the cubic terms are orders of
    # magnitude larger than the linear one and the Hessian is singular without it.
    return (basis - basis.mean(axis=0)) / basis.std(axis=0)


def zscore(a):
    return (a - a.mean(axis=0)) / a.std(axis=0)


def adjusted_fit(design):
    return sm.Logit(y_ff, sm.add_constant(design)).fit(disp=0, method="bfgs", maxiter=3000)


def other_predictors(var):
    return zscore(df_pilot[[c for c in feature_cols if c != var]]
                  .astype(float).to_numpy())


def scales_of(var):
    return {
        "raw": df_pilot[var].astype(float).to_numpy(),
        "Yeo-Johnson (as fitted)": PowerTransformer("yeo-johnson").fit_transform(
            df_pilot[[var]].astype(float)).ravel(),
    }


header = (f"{'Predictor':<21}{'Scale':<26}{'LR chi2':>9}{'df':>4}{'p':>9}"
          f"{'AIC linear':>12}{'AIC spline':>12}")
print(header)
print("-" * len(header))

for var in CONTINUOUS:
    others = other_predictors(var)
    for label, v in scales_of(var).items():
        linear = adjusted_fit(np.column_stack([zscore(v), others]))
        basis = rcs_basis(v, np.quantile(v, KNOT_QUANTILES[4]))
        spline = adjusted_fit(np.column_stack([basis, others]))
        lr_stat = 2 * (spline.llf - linear.llf)
        df_free = basis.shape[1] - 1
        p_value = 1 - chi2.cdf(lr_stat, df_free)
        print(f"{var:<21}{label:<26}{lr_stat:>9.2f}{df_free:>4d}{p_value:>9.4f}"
              f"{linear.aic:>12.1f}{spline.aic:>12.1f}")


# ── How many knots does age need? ────────────────────────────────────────────
print("\n\nAge - how many knots? (raw scale, otherwise the full pilot model)")
print(f"{'Specification':<26}{'extra params':>13}{'AIC':>9}{'LR vs linear':>14}{'p':>9}")
print("-" * 71)
age = df_pilot["LEEFTIJD_EXACT"].astype(float).to_numpy()
others_age = other_predictors("LEEFTIJD_EXACT")
linear_age = adjusted_fit(np.column_stack([zscore(age), others_age]))
print(f"{'linear':<26}{0:>13d}{linear_age.aic:>9.1f}{'-':>14}{'-':>9}")
for n_knots, quantiles in KNOT_QUANTILES.items():
    basis = rcs_basis(age, np.quantile(age, quantiles))
    fit = adjusted_fit(np.column_stack([basis, others_age]))
    lr_stat = 2 * (fit.llf - linear_age.llf)
    df_free = basis.shape[1] - 1
    print(f"{f'restricted cubic spline, {n_knots} knots':<26}{df_free:>13d}{fit.aic:>9.1f}"
          f"{lr_stat:>14.2f}{1 - chi2.cdf(lr_stat, df_free):>9.4f}")


# ── Observed risk against each predictor (unadjusted, for shape only) ─────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
labels = ["Age (years)", "BMI (kg/m2)", "Injury Severity Score", "Number of fractures"]
for ax, var, xlabel in zip(axes, CONTINUOUS, labels):
    v = df_pilot[var].astype(float).to_numpy()
    grid = np.linspace(np.quantile(v, 0.01), np.quantile(v, 0.99), 100)
    ax.plot(grid, lowess(y_ff, v, frac=0.6, it=0, xvals=grid),
            color="crimson", linewidth=2, label="LOWESS")

    deciles = pd.qcut(v, 10, duplicates="drop")
    observed = (pd.DataFrame({"v": v, "y": y_ff})
                .groupby(deciles, observed=True)
                .agg(mid=("v", "median"), rate=("y", "mean")))
    ax.plot(observed["mid"], observed["rate"], "o",
            color="steelblue", label="Decile observed risk")

    ax.axhline(y_ff.mean(), color="grey", linestyle=":", linewidth=1,
               label="Cohort rate (13.3%)")
    ax.set_xlabel(xlabel)
    ax.set_title(xlabel, fontsize=11)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("30-day mortality")
axes[0].legend(fontsize=8)

fig.suptitle("Functional form of the continuous predictors", fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/functional_form_continuous.png", dpi=150)
plt.show()

#### Result

| Predictor | Scale | LR chi2 (df=2) | p | AIC linear | AIC spline |
|---|---|---|---|---|---|
| `LEEFTIJD_EXACT` | raw | 27.03 | <0.0001 | 690.7 | **667.7** |
| `LEEFTIJD_EXACT` | Yeo-Johnson (as fitted) | 28.53 | <0.0001 | 692.5 | **668.0** |
| `BMI` | raw | 3.39 | 0.18 | **690.7** | 691.3 |
| `BMI` | Yeo-Johnson (as fitted) | 1.44 | 0.49 | **690.2** | 692.8 |
| `ISS08` | raw | 5.60 | 0.061 | **690.7** | 689.1 |
| `ISS08` | Yeo-Johnson (as fitted) | 7.40 | 0.025 | 691.7 | **688.3** |
| `NUMBER_OF_FRACTURES` | raw | 1.63 | 0.44 | **690.7** | 691.1 |
| `NUMBER_OF_FRACTURES` | Yeo-Johnson (as fitted) | 0.69 | 0.71 | **690.1** | 691.4 |

**Age is clearly non-linear, and the Yeo-Johnson transform does not fix it.** The likelihood
ratio test rejects linearity decisively on both scales, and the spline lowers AIC by roughly 23
points - far beyond the 2 points that would make a 2-parameter extension break even. That the
test is, if anything, *slightly stronger* on the Yeo-Johnson scale is the point: the transform
is monotone, so it can stretch the age axis but cannot introduce the change of direction the
data show.

The shape is visible in the left-hand panel. Observed 30-day mortality is flat to mildly
declining from age 18 to about 45 (8.7%, 7.4%, 7.4%, 3.8% across the first four deciles), drifts
up through middle age, and then climbs steeply: 25.0% in the ninth decile (median age 76) and
36.2% in the tenth (median age 83). A single slope through that pattern under-predicts risk in
the very old and over-predicts it in the young - exactly where a triage model is used.

**BMI is adequately linear.** Neither test reaches significance and the spline makes AIC worse on
both scales, so the extra parameters buy nothing. Restricted to the 1,284 patients with a
recorded BMI, observed mortality across deciles stays between 6% and 15% with no trend
(11, 9, 14, 8, 15, 12, 12, 12, 6, 12 percent from lowest to highest decile). The U-shaped
obesity-paradox relationship reported in some trauma cohorts is not detectable here; the
near-zero BMI coefficient in the pilot model is a genuinely weak association, not a linear fit
masking a curve.

> Read the two extreme decile points near BMI 25 in the BMI panel as an artefact, not a signal.
> This section runs on the median/mode-imputed pilot frame of section 11, so the 202 patients
> with a missing BMI all sit at the median (25.2), putting 209 patients on one value. The decile
> boundaries then cut that pile arbitrarily, which is what produces the 24% and 4% points sitting
> on top of each other. The complete-case deciles quoted above are the ones to trust, and the
> likelihood-ratio tests are unaffected because they use BMI continuously.

**ISS08 is borderline, and left linear.** This is the one genuinely ambiguous case. On the
raw scale the test misses conventional significance (chi2 = 5.60, p = 0.061); on the Yeo-Johnson
scale it clears it (chi2 = 7.40, p = 0.025). Four predictors were tested here, so a single
p = 0.025 is not strong evidence on its own - a Bonferroni adjustment across the four puts it at
0.10. More to the point, every ISS specification lands within about 2.5 AIC points of every
other (690.7 raw linear, 691.7 Yeo-Johnson linear, 689.1 raw spline, 688.3 Yeo-Johnson spline),
against the 23-point gap that made the case for splining age. Adding 2 parameters to move AIC by
2 is not a trade worth making, so ISS08 stays a single linear term.

The decile pattern shows why the test flickers: observed mortality is flat and low up to about
ISS 20 (5%, 5%, 1%, 5%, 9%) and then rises sharply (21%, 19%, 29%, 50%). That is real curvature,
but the Yeo-Johnson transform with lambda = 0.12 - close to a log - already absorbs most of it,
which is exactly what it is there for. Worth noting for the record that raw linear ISS beats
Yeo-Johnson linear ISS by 1.0 AIC point: the transform is not earning its place for this
predictor, it is simply not costing much either.

> If a reviewer presses on this, the defensible alternative is ISS08 with a 3-knot spline. It is
> recorded here as a considered and rejected option, not an oversight.

**NUMBER_OF_FRACTURES is adequately linear.** Neither test comes close (p = 0.44 raw,
p = 0.71 transformed) and the spline worsens AIC on both scales. It stays a single linear
term; section 6e sets its final coding (capped at 6+, untransformed).

#### How many knots for age?

Knot count is a genuine choice, so it is made explicitly rather than left at a default.
All placements are at fixed quantiles of the age distribution - never chosen by scanning for the
best fit, which would be selecting on the outcome and would leave the bootstrap unable to see
the optimism it creates.

| Specification | Extra parameters | AIC | LR vs linear | p |
|---|---|---|---|---|
| linear | 0 | 690.7 | - | - |
| **restricted cubic spline, 3 knots** | **1** | **667.2** | **25.57** | **<0.0001** |
| restricted cubic spline, 4 knots | 2 | 667.7 | 27.03 | <0.0001 |
| restricted cubic spline, 5 knots | 3 | 663.6 | 33.13 | <0.0001 |

**Three knots wins.** It costs a single extra parameter and already recovers 25.6 of the 33.1
chi-square available at 5 knots, giving a *better* AIC than the 4-knot fit. Five knots buys
3.6 more AIC points for 2 more parameters, and places knots in the sparse tails of the age
distribution where a spline is least stable - Harrell's guidance is 3 knots when events are
limited, and 198 events with 24 other parameters qualifies.

#### Decision

> **Implemented.** `LEEFTIJD_EXACT` enters the model as a restricted cubic spline with 3
> knots at the 10th, 50th and 90th percentiles, configured under `features.spline` in
> `config/pipeline.yaml` and expanded by `Transformer`. Knots are refitted on every bootstrap
> resample and every cross-validation fold along with everything else, so the optimism-corrected
> estimates account for the spline exactly as they account for the imputer and the scaler.
>
> `BMI` and `ISS08` keep their single linear term with the Yeo-Johnson transform, unchanged.
> `NUMBER_OF_FRACTURES` keeps its single linear term, capped at 6+ and untransformed (6e).
>
> Net cost is 1 parameter: 24 to 25. Section 11b puts the cross-validated R2_cs at 0.243, which
> by Riley criterion B3 supports up to 46, so the margin is unaffected.
>
> **Reporting consequence.** A spline has no single odds ratio, so age can no longer appear as a
> row in the coefficient table. It is reported instead as an odds-ratio curve against a reference
> age of 50 with a bootstrap interval (`outputs/explainability/spline_LEEFTIJD_EXACT.png`) plus a
> table of point estimates at 30-90 years. The two basis columns are excluded from the forest
> plot, since neither is interpretable on its own.

<a id="sec12"></a>

## 12. Summary and conclusions

### Cohort
N = 1,486 included patients (31 excluded), accident years 2018-2024. Enrollment is stable
(195-236 per year) and 30-day mortality shows no monotone drift (11.3%-16.8% by year).
30-day mortality: **13.3%** (198 events). Median time to death among patients who died within
one year (n = 234): **4 days**.

### Data quality
- **The AO N-modifier (`N_CODE`) was not collected in 2019-2020.** 423/1,486 rows (28.5%) carry an
  out-of-schema `0`, present in 100% of 2019 and 2020 records and 0% of every other year. See section 8.
- **`ASIA` is unrecorded for 118 patients, all of them `N_CODE` = NX**, with 30-day mortality of
  80.5% vs 7.5%. This is a not-examinable state, not ordinary missingness, and is now carried by
  the derived `NEURO_NOT_EXAMINABLE` indicator rather than imputed. See section 8.

### Sample size
Pilot model (median/mode imputation, N = 1,486, E = 198, P = 23 parameters):

| Estimate | R2_cs | R2_Nagelkerke | Max P at <=10% shrinkage |
|---|---|---|---|
| Apparent (in-sample) | 0.297 | 0.546 | 59 |
| 5-fold cross-validated | 0.243 | 0.447 | 46 |

Apparent optimism is 0.054 R2_cs units. The cross-validated estimate is the appropriate input
for the decision and permits up to **46** parameters; the candidate set uses **23**, so the
shrinkage criterion is satisfied with a wide margin.

For context, 23 parameters against 198 events is **8.6 events per parameter**. That is below
the traditional rule of thumb of 10, which the Riley framework was explicitly developed to
replace: the rule of thumb ignores how much signal the predictors actually carry, and at an
R2_cs of 0.24 this sample supports considerably more parameters than the rule would allow.

### Missingness
| Variable | Missing | Note |
|---|---|---|
| BMI | 202 (13.6%) | only variable with substantial missingness |
| ASIA | 117 (7.9%) | not-examinable state, not MAR - handled by `NEURO_NOT_EXAMINABLE`, which flags 118 patients: these 117 plus one recorded as `0` = not determinable |
| COMORB | 54 (3.6%) | |
| INTENTIE | 4 (0.3%) | unmapped / blank |
| NUMBER_OF_FRACTURES, AMOUNT_OF_LEVELS_INJURED | 1 each | |

All other candidate predictors are complete. No variable approaches the 30% threshold that would
warrant exclusion on missingness grounds.

Little's MCAR test across the nine partially observed numeric columns rejects MCAR
(chi-square = 425.5, df = 123, p < 0.001), so complete-case analysis would be biased.

`BMI` missingness is strongly associated with 30-day death - 28.3% missing among patients who
died versus 11.3% among survivors (OR 3.97, 95% CI 2.58-6.13, p < 0.001) - and weakly with older
age (OR 1.01 per year, p = 0.021) and *lower* ISS (OR 0.98 per point, p = 0.002). McFadden
pseudo-R2 = 0.049. Because missingness is predictable from observed variables it is MAR, which
supports imputation over listwise deletion. See section 3b.

### Predictor signal
Pilot multivariate ORs, significant at p < 0.05:

| Predictor | OR (95% CI) | p |
|---|---|---|
| NEURO_NOT_EXAMINABLE | 35.19 (18.75-66.03) | <0.001 |
| COMORB (per ASA grade) | 2.02 (1.47-2.77) | <0.001 |
| ISS08 (per point) | 1.08 (1.06-1.10) | <0.001 |
| LEEFTIJD_EXACT (per year) | 1.04 (1.02-1.05) | <0.001 |
| L (lumbar involvement) | 0.45 (0.23-0.88) | 0.019 |



### Feature selection decisions
| Dropped | Reason | Kept instead |
|---|---|---|
| SCI\_yes\_no | Near-perfect collinearity with ASIA (r = -0.955 among examinable patients) | ASIA |
| N\_CODE | r = -0.93 with ASIA among validly coded rows; not collected in 2019-2020; unique signal (NX) is now explicit | ASIA + NEURO\_NOT\_EXAMINABLE |
| AMOUNT\_OF\_LEVELS\_INJURED, n\_regions | Collinear with NUMBER\_OF\_FRACTURES (r = 0.76 / 0.70) | NUMBER\_OF\_FRACTURES |
| HT\_CODE, CHF\_CODE, COPD\_CODE, DM\_CODE, DEPENDENT\_CODE | The five items that MFI5 is the sum of; the index replaces them | MFI5 |
| ASA, ASAcat, FrailtyCat | ASA is blank or non-informative for 1,137/1,486; ASAcat and FrailtyCat are coarsened versions of COMORB and MFI5 | COMORB, MFI5 |

`ASIA_exam`, `N_CODE_valid` and `SCI_yes_no` are mutually collinear (-0.96, -0.93, 0.90):
one neurological severity construct measured three ways. `ASIA` is kept as that construct,
with `NEURO_NOT_EXAMINABLE` carrying the state the other two conflated with severity.

### Predictor coding decisions

| Predictor | Coding | Why | Section |
|---|---|---|---|
| `LEEFTIJD_EXACT` | restricted cubic spline, 3 knots | linear term rejected, LR chi2 = 25.6 on 1 df, AIC 690.7 -> 667.2; Yeo-Johnson is monotone and cannot fix it | 11c |
| `BMI`, `ISS08` | linear + Yeo-Johnson | linearity not rejected (ISS08 borderline at p = 0.061 raw, rejected on AIC grounds) | 11c |
| `NUMBER_OF_FRACTURES` | ordinal, capped at 6+, untransformed | linearity not rejected; the cap limits the sparse tail and gives an odds ratio per fracture | 6e, 11c |
| `MFI5` | ordinal, capped at 3+ | grades 4 and 5 together hold fewer than 10 patients; capping restores a monotone gradient | 6d |
| `INTENTIE_cat` | 6 levels + explicit `unknown` | nominal, so a missing mechanism cannot be imputed on a scale; 4 patients | 6c |
| `ASIA` | ordinal, with `NEURO_NOT_EXAMINABLE` | the not-examinable state is prognostic, not missing | 8 |

### Final candidate feature set - 19 features (25 model parameters)

| Type | Count | Features |
|---|---|---|
| Continuous, linear | 2 | BMI, ISS08 |
| Continuous, splined | 1 | LEEFTIJD\_EXACT (2 basis columns) |
| Binary | 11 | Geslacht01, TBI\_CT, TYPE\_A/B/C/F\_INJURY, C, T, L, S, NEURO\_NOT\_EXAMINABLE |
| Ordinal | 4 | COMORB, ASIA, MFI5 (capped at 3+), NUMBER\_OF\_FRACTURES (capped at 6+) |
| Categorical | 1 | INTENTIE\_cat (6 dummies incl. `unknown`, reference: fall\_low) |

Matches `features:` in `config/pipeline.yaml`. The sample-size headroom is unchanged: 25
parameters against the 46 that Riley criterion B3 permits at the observed cross-validated
R2_cs of 0.243.
